# [1.5.1] Balanced Bracket Classifier (연습 문제)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/31_[1.5.1]_Balanced_Bracket_Classifier)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part51_balanced_bracket_classifier/1.5.1_Balanced_Bracket_Classifier_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part51_balanced_bracket_classifier/1.5.1_Balanced_Bracket_Classifier_solutions.ipynb?t=20260329)**

문제나 버그가 있다면 [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA)의 `#errata` 채널로 보내주시고, 본 장의 학습 내용에 관한 질문은 전용 채널에서 해주시기 바랍니다.

마크다운 헤더 셀 왼쪽에 있는 화살표 기호를 클릭하면 각 섹션을 접어서 헤더만 보이게 할 수 있습니다.

다른 모든 장으로 이동하는 링크: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

> *참고 - 어느 시점에서든 numpy 관련 에러가 발생하면 (예: import 셀을 처음 실행할 때 또는 첫 번째 numpy 함수를 실행할 때), 커널을 재시작하고 설정 코드를 다시 실행해야 합니다. 그러면 에러가 해결될 것입니다.*

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-15-1.png" width="350">

# 소개

모델이 합성된 알고리즘 태스크로 학습될 때, 내부적으로는 종종 깔끔하고 해석 가능한 계산 방식을 학습합니다. 적절한 태스크를 선택하고 모델을 역공학(reverse engineer)하려는 시도는 해석할 만한 흥미로운 circuit들이 많은 풍부한 영역이 될 수 있습니다! 어떤 의미에서 이는 '쉬운 모드'의 interpretability입니다. 모델은 보통 단일 태스크로 학습되며(모든 언어에 대해 배워야 하는 language model과 달리), 데이터에 대한 정확한 ground truth와 최적의 솔루션을 알고 있고, 모델의 크기도 매우 작기 때문입니다. 그렇다면 왜 이것에 관심을 가져야 할까요?

알고리즘 문제를 다루는 것은 우리에게 다음과 같은 기회를 제공합니다:

* interpretability를 연습하고, 직관을 기르며, 기법들을 학습할 수 있습니다.
* ground truth가 잘 알려진 문제에 적용해 봄으로써, 적절한 도구와 기법에 대한 이해를 정교화할 수 있습니다.
* 특히 흥미로운 종류의 행동을 격리하여 상세히 연구하고 더 잘 이해할 수 있습니다 (예: Anthropic의 [Toy Models of Superposition](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=EuO4CLwSIzX7AEZA1ZOsnwwF) 논문).
* 작은 모델을 역공학하며 얻은 통찰을 바탕으로, 어떤 결과가 일반화될 수 있는지, 또는 circuit를 식별하는 데 사용한 기법 중 자동화하여 대규모로 사용할 수 있는 것이 있는지 조사할 수 있습니다.

이 연습 문제들에서 다룰 알고리즘 문제는 **bracket classification**입니다. 즉, `"(())()"`과 같은 괄호 문자열을 입력받아 "balanced" 또는 "unbalanced"라는 예측값을 출력하는 것입니다. 우리는 이 문제를 해결하기 위한 알고리즘적 솔루션을 찾고, 이 알고리즘의 일부를 구현하는 모델 내의 circuit 중 하나를 역공학할 것입니다.

이 페이지에는 많은 수의 연습 문제가 포함되어 있습니다. 각 연습 문제에는 5점 만점의 난이도와 중요도 등급, 그리고 권장 최대 소요 시간과 때로는 짧은 주석이 달려 있습니다. 등급과 예상 시간은 상대적으로 해석하시기 바랍니다 (예: 예상 시간보다 약 50% 더 많은 시간을 쓰고 있다면, 그에 맞춰 조정하십시오). 충분히 중요하지 않다고 느껴지거나 더 핵심적인 내용으로 빠르게 넘어가고 싶다면, 연습 문제를 건너뛰거나 솔루션을 확인하셔도 좋습니다!

## 동기

A [Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html)에서 우리는 toy language model을 해석하며 많은 성과를 거두었습니다. toy language model이란 더 큰 모델들과 정확히 동일한 방식으로 학습되었지만, 레이어가 1개 또는 2개뿐인 transformer를 의미합니다. toy language model을 연구할 때 여전히 얻어낼 수 있는 쉬운 성과들이 많이 남아 있는 것으로 보입니다!

그렇다면 왜 toy language model 연구에 관심을 가져야 할까요? 분명한 이유는 **성과를 내기가 훨씬 쉽기 때문**입니다. 특히, 모델의 [inputs and outputs](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=UriJZK6E8dnL8NDY-fGl_eFX)은 본질적으로 해석 가능하며, toy model에서는 입력과 출력 사이에 이상한 복잡성이 쌓일 만한 공간이 그리 많지 않습니다. 하지만 이에 대한 당연한 반론은, 궁극적으로 우리가 관심을 갖는 것은 실제 모델(이상적으로는 GPT-3와 같은 매우 큰 모델)을 이해하는 것이며, toy model을 해석하는 법을 배우는 것이 실제 목표는 아니라는 점입니다. 이는 상당히 타당한 반론이지만, toy model 연구가 가치 있을 수 있는 두 가지 자연스러운 방법이 있습니다.

첫 번째는 더 큰 모델에서도 반복해서 나타나는 근본적인 circuit을 찾고, 더 큰 모델에서 이러한 circuit을 쉽게 식별할 수 있게 해주는 [motifs](https://distill.pub/2020/circuits/zoom-in/#claim-2-motifs)를 찾는 것입니다. 여기서 핵심적인 근본 질문은 [universality](https://distill.pub/2020/circuits/zoom-in/#claim-3)에 관한 것입니다. 즉, 각 모델이 작업을 완수하기 위해 자신만의 이상한 방식을 학습하는 것일까요, 아니면 모든 모델이 수렴하는 몇 가지 근본적인 원리와 알고리즘이 존재하는 것일까요?

두 번째는 모델을 역공학(reverse engineer)하는 방법에 대해 더 나은 이해를 형성하는 것입니다. 어떤 직관과 개념적 프레임워크가 적절한지, 어떤 도구와 기술이 작동하고 작동하지 않는지, 그리고 우리가 어떤 이상한 한계에 직면할 수 있는지를 파악하는 것입니다. 예를 들어, A Mathematical Framework의 연구는 [the residual stream as the central object](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=DHp9vZ0h9lA9OCrzG2Y3rrzH)와 같은 아이디어, 그리고 [QK-Circuits](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=n_Lc0Z5N9HMhAYcycDda-UEB) 및 [OV-Circuits](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=n_Lc0Z5N9HMhAYcycDda-UEB)의 중요성을 제시하며, 이는 많은 서로 다른 모델로 일반화되는 것으로 보입니다. 또한 이후 연습 문제에서 MLP가 어떻게 서로 다른 feature에 반응하여 활성화되는 neuron들의 집합으로 생각될 수 있는지 보여주는 사례를 살펴볼 것입니다. 이는 많은 language model에서 나타나는 현상과 유사합니다. 하지만 오해의 소지가 있는 부분도 있으며, toy model에서 잘 작동하는 일부 기술들은 일반화가 덜 되는 것처럼 보이기도 합니다.

## 이 실습들의 목적 / 구조

표면적으로 이 실습들은 괄호 분류(bracket classification)로 학습된 bidirectional 모델의 부분적인 해석 과정을 안내하도록 설계되었습니다. 하지만 동시에 여러분을 더 뛰어난 interpretability 연구자로 만들기 위해 설계되었습니다! 결과적으로, 대부분의 실습은 다음 사항들의 조합으로 구성됩니다:

1. circuit의 새로운 feature/component를 보여주는 것, 그리고
2. 더 넓은 mech interp 맥락에서 도구를 사용하고 결과를 해석하는 방법을 가르치는 것입니다.

이 실습들을 진행하다 보면, 구현하고 있는 기술의 까다로운 세부 사항이나 계산하고 있는 내용에 매몰되기 쉽습니다. 현재 어떤 질문을 던지려 하는지, 얻은 출력을 어떻게 해석할 것인지, 그리고 현재 사용 중인 도구들이 모델에 대한 더 나은 이해를 돕기 위해 어떻게 가이드하고 있는지 스스로 질문하며 계속해서 높은 수준의 관점을 유지하시기 바랍니다.

## 콘텐츠 및 학습 목표

### 1️⃣ Bracket classifier

이 섹션에서는 transformer를 분류(classification)에 어떻게 사용할 수 있는지, 그리고 TransformerLens에서 이것이 어떻게 작동하는지(permanent hook 사용)에 대한 세부 사항을 설명합니다. 또한 balanced brackets 문제에 대한 해결책을 직접 작성하는 실습을 진행합니다.

*이 섹션은 주로 기초를 다지는 단계이며, 내용은 매우 가볍습니다.*

> ##### 학습 목표
>
> * transformer를 classification에 어떻게 사용할 수 있는지 이해합니다.
> * TransformerLens의 permanent hook을 통해 특정 종류의 transformer 동작(예: padding token의 masking)을 구현하는 방법을 이해합니다.
> * transformer의 inductive bias를 고려할 때, 이러한 문제들에 대해 transformer가 어떤 종류의 알고리즘적 해결책을 찾을 가능성이 높은지 생각하기 시작합니다.

### 2️⃣ Moving backwards

여기서는 logit attribution을 수행하고, 모델의 특정 경로를 역추적하여 최종 classification 확률에 어떤 컴포넌트가 가장 중요한지 파악하는 방법을 배웁니다.

모델에서 LayerNorm을 다루게 되는 첫 번째 단계입니다.

*induction head에 대해 logit attribution을 수행해 본 적이 있다면 이 섹션이 익숙할 것입니다 (다만, 이번 실습은 코딩 관점에서 약간 더 어렵습니다). LayerNorm 기반 실습은 조금 까다로울 수 있습니다!*

> ##### 학습 목표
>
> * logit attribution을 수행하는 방법을 이해합니다.
> * 모델을 역추적하여 최종 classification 확률에 가장 중요한 컴포넌트를 식별하는 방법을 이해합니다.
> * LayerNorm이 어떻게 작동하는지 이해하고, 모델에서 이를 처리하는 몇 가지 방법을 살펴봅니다.

### 3️⃣ Total elevation circuit

*이 섹션은 코딩과 개념적 관점 모두에서 상당히 도전적입니다. 관찰과 개입(intervention)의 결과를 모델이 어떻게 작동하는지에 대한 구체적인 가설과 연결해야 하기 때문입니다.*

실습의 가장 큰 비중을 차지하는 이 섹션에서, 여러분은 서로 다른 head의 attention pattern을 조사하고, 이를 인간이 이해할 수 있는 알고리즘(예: copying 또는 aggregation)으로 해석합니다. 관찰 결과를 바탕으로, 특정 유형의 balanced brackets 실패 모드(왼쪽과 오른쪽 괄호의 개수 불일치)가 모델에 의해 어떻게 감지되는지 추론합니다.

모델에서 MLP를 다루게 되는 첫 번째 단계입니다.

> ##### 학습 목표
>
> * 독특한 attention pattern을 인간이 이해할 수 있는 알고리즘과 연결하고, 모델 동작에 대해 추론하는 연습을 합니다.
> * MLP를 neuron의 집합으로 보는 방법을 이해합니다.
> * total elevation circuit의 전체적인 모습과 작동 방식에 대한 완전한 그림을 구축합니다.

### ☆ Bonus exercises

마지막으로, 이전 내용을 바탕으로 하는 몇 가지 선택적 보너스 실습이 있습니다 (예: 모델의 다른 부분을 조사하거나, 모델 작동 방식에 대한 이해를 바탕으로 adversarial examples를 생성하는 것).

*이 마지막 섹션은 가이드가 적은 편이지만, 제안된 실습들은 이전 섹션과 비슷한 성격입니다.*

> ##### 학습 목표
>
> * 모델 작동 방식에 대한 이해를 사용하여 adversarial examples를 생성합니다.
> * 모델의 특정 변칙적인 feature들을 더 깊게 파고듭니다.

## 설정 코드

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install einops jaxtyping transformer_lens==2.17.0 git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import json
import sys
from functools import partial
from pathlib import Path

import circuitsvis as cv
import einops
import torch as t
from IPython.display import display
from jaxtyping import Bool, Float, Int
from sklearn.linear_model import LinearRegression
from torch import Tensor, nn
from tqdm import tqdm
from transformer_lens import ActivationCache, HookedTransformer, HookedTransformerConfig, utils
from transformer_lens.hook_points import HookPoint

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
t.set_grad_enabled(False)

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part51_balanced_bracket_classifier"
exercises_dir = next(p for p in Path.cwd().parents if p.name == chapter) / "exercises"
section_dir = exercises_dir / section

import part51_balanced_bracket_classifier.tests as tests
import plotly_utils
from part51_balanced_bracket_classifier.brackets_datasets import BracketsDataset, SimpleTokenizer
from plotly_utils import bar, hist

MAIN = __name__ == "__main__"

# 1️⃣ Bracket classifier

> ##### 학습 목표
>
> * transformer가 classification에 어떻게 사용될 수 있는지 이해합니다.
> * TransformerLens의 permanent hook을 통해 특정 종류의 transformer 동작(예: padding token의 masking)을 구현하는 방법을 이해합니다.
> * transformer의 inductive bias를 고려할 때, 이러한 문제들에 대해 transformer가 어떤 종류의 알고리즘적 해결책을 찾을 가능성이 높은지 생각하기 시작합니다.

이 섹션에서는 transformer를 classification에 어떻게 사용할 수 있는지, 그리고 TransformerLens에서 이것이 어떻게 작동하는지(permanent hook 사용)에 대한 세부 사항을 설명합니다. 또한 balanced brackets 문제에 대한 솔루션을 직접 작성하는 실습을 진행합니다.

*이 섹션은 주로 기초를 다지는 단계이며, 내용은 매우 가볍습니다.*

---

대규모 언어 모델이 학습하는 많은 동작 중 하나는 중첩된 괄호 시퀀스가 균형 잡혀 있는지 판단하는 능력입니다. 예를 들어, `(())()`, `()()`, `(()())`는 균형 잡힌 시퀀스인 반면, `)()`, `())()`, `((()((())))`는 그렇지 않습니다.

학습 과정에서 균형 잡힌 괄호를 포함한 텍스트는 불균형한 괄호를 포함한 텍스트보다 훨씬 더 흔하게 나타납니다. 특히 GitHub에서 수집한 소스 코드는 대부분 구문적으로 유효합니다. 따라서 "다음 token을 예측하라"와 같은 pretraining 목적 함수는 시퀀스가 불균형할 때 닫는 괄호가 나올 가능성이 더 높고, 현재 시퀀스가 균형 잡혀 있다면 그 가능성이 매우 낮다는 것을 모델이 학습하도록 유도합니다.

우리가 답을 얻고 싶은 몇 가지 질문은 다음과 같습니다:

- 이 동작은 얼마나 강건(robust)합니까? 어떤 입력에서 실패하며 그 이유는 무엇입니까?
- 이 동작은 distribution 외부로 어떻게 일반화됩니까? 예를 들어, 학습 과정에서 보지 못한 중첩 깊이나 시퀀스 길이를 처리할 수 있습니까?

만약 모델을 black box 함수로 취급하고 모델이 생성하는 입력/출력 쌍만 고려한다면, 많은 연산 자원을 사용하여 수많은 입력을 확인하더라도 그 동작에 대해 보장할 수 있는 것은 매우 제한적입니다. 이것이 interpretability의 동기가 됩니다. 내부 구조를 파헤침으로써 이러한 질문들에 대한 통찰을 얻을 수 있을까요? 만약 모델이 강건하지 않다면, 모델이 잘못된 예측을 확신하게 만드는 adversarial examples를 직접 찾아낼 수 있을까요? 함께 알아봅시다!

## 오늘의 Toy Model

오늘은 괄호 시퀀스가 균형 잡혀 있는지(balanced) 여부만을 분류하도록 학습된 작은 transformer를 공부하겠습니다. 이 모델은 크기가 작아서 실험을 빠르게 수행할 수 있지만, 해당 태스크를 충분히 잘 수행할 수 있을 만큼은 큽니다. 가중치와 architecture는 제공됩니다.

### Causal vs bidirectional attention

이 모델과 여러분이 이미 구현해 보았을 GPT 스타일 모델의 핵심적인 차이점은 attention 메커니즘입니다.

GPT는 **causal attention**을 사용하며, 여기서 attention score는 source token이 destination token보다 뒤에 오는 모든 위치에서 마스킹됩니다. 이는 정보가 모델 내에서 앞으로만 흐를 수 있고, 절대 뒤로 흐를 수 없음을 의미합니다 (이 덕분에 모델을 병렬로 학습시킬 수 있습니다. 모델의 출력은 다음 token에 대한 일련의 분포이며, 각 분포는 오직 이전에 등장한 token들의 정보만 사용할 수 있습니다). 반면 이 모델은 **bidirectional attention**을 사용하며, attention score가 source와 destination token의 상대적 위치에 따라 마스킹되지 않습니다. 이는 정보가 양방향으로 흐를 수 있으며, 모델이 과거를 예측하기 위해 미래의 정보를 사용할 수 있음을 의미합니다.

### 분류를 위한 transformer 사용하기

GPT는 다음 token에 대한 예측값과 실제 다음 token 사이의 cross-entropy loss에 대해 gradient descent를 통해 학습됩니다. 분류를 수행하도록 설계된 모델들도 매우 유사한 방식으로 학습되지만, 다음 token에 대한 확률 분포를 출력하는 대신 클래스 레이블에 대한 분포를 출력합니다. 우리는 `[d_model, num_classifications]` 크기의 unembedding matrix를 가지고, 단 하나의 시퀀스 위치(보통 0번째 위치)만을 사용하여 분류 확률을 표현함으로써 이를 구현합니다.

아래는 모델 architecture와 사용 방식의 차이를 비교한 도식입니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/gpt-vs-bert-last.png" width="1250">

다른 모든 시퀀스 위치의 출력이 버려진다고 해서, 그 위치들이 유용하지 않다는 뜻은 아닙니다. 그 위치들은 거의 확실하게 중요한 중간 계산이 일어나는 지점이 될 것입니다. 하지만 이는 모델이 분류를 위해 정보를 사용하려면, 반드시 해당 위치들에서 0번째 위치로 정보를 이동시켜야 함을 의미합니다.

### softmax에 관한 참고 사항

각 괄호 시퀀스에 대해, 우리의 (중요한) 출력은 두 개의 값으로 이루어진 벡터 `(l0, l1)` 이며, 이는 (unbalanced, balanced)에 대한 모델의 logit 분포를 나타냅니다. 우리 모델은 이 logit들과 실제 레이블 사이의 cross-entropy loss를 최소화함으로써 학습되었습니다. 흥미롭게도, logit은 translation invariant하므로 우리가 실제로 신경 쓰는 유일한 값은 logit 간의 차이인 `l0 - l1` 입니다. 이는 시퀀스가 balanced인 경우 대비 unbalanced일 확률의 log likelihood ratio입니다. 나중에 우리는 이 `logit_diff` 를 사용하여 모델에서 logit attribution을 수행할 수 있을 것입니다.

### padding token 마스킹하기

우측 상단의 이미지는 사실 약간 불완전합니다. 모델이 서로 다른 길이의 시퀀스를 어떻게 처리하는지 보여주지 않기 때문입니다. 결국 학습 과정에서 모든 시퀀스를 하나의 tensor로 묶어 배치(batch) 처리하려면 모든 시퀀스의 길이가 같아야 합니다. 모델은 이를 위해 두 가지 새로운 token인 end token과 padding token을 통해 관리합니다.

end token은 모든 괄호 시퀀스의 끝에 위치하며, 그 후 시퀀스가 특정 고정 길이에 도달할 때까지 끝에 padding token을 추가합니다. 예를 들어, 이 모델은 최대 길이 40의 괄호 시퀀스로 학습되었으므로, 만약 괄호 문자열 `(())` 을 분류하고 싶다면 이를 길이 42의 시퀀스로 패딩합니다:

```
[start] + ( + ( + ) + ) + [end] + [pad] + [pad] + ... + [pad]
```

attention score를 계산할 때, key가 padding token인 모든 (query, key) 위치에서 마스킹을 수행합니다. 이는 GPT의 causal masking이 미래 token에서 과거 token으로 정보가 흐르지 않게 하는 것과 마찬가지로, padding token에서 시퀀스의 다른 token으로 정보가 흐르지 않도록 보장합니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/gpt-vs-bert-4.png" width="950">

query가 padding token이고 key가 padding token이 아닐 때는 attention score가 마스킹되지 않는다는 점에 유의하십시오. 이론적으로 이는 정보가 padding token 위치에 저장될 수 있음을 의미합니다. 하지만 padding token key 위치는 항상 마스킹되기 때문에, 이 정보는 다시 시퀀스의 나머지 부분으로 흐를 수 없으며, 따라서 최종 출력에 영향을 주지 않습니다. (또한, query 위치까지 마스킹한다면 모든 요소가 마이너스 무한대인 행에 대해 softmax를 취하게 되어 수치적 오류가 발생하며, 이는 정의되지 않은 연산이라는 점에 유의하십시오!)

<details>
<summary> <b>BERT</b> </summary>와의 관련성에 대하여

이 모든 과정은 bidirectional transformer인 **BERT**가 작동하는 방식과 매우 유사합니다:

* BERT는 `[start]` 대신 `[CLS]` (classification) token을 가지지만, 작동 방식은 정확히 같습니다.
* BERT는 `[end]` 대신 `[SEP]` (separation) token을 가지며, 이는 유사한 기능을 수행하지만 **NSP** (next sentence prediction)에서 사용될 때 특별한 목적을 가집니다.

이에 대해 더 자세히 읽어보고 싶으시다면 [this link](https://albertauyeung.github.io/2020/06/19/bert-tokenization.html/) 을 확인하시기 바랍니다.

</details>

우리는 TransformerLens의 **permanent hooks** 기능을 사용하여 이러한 유형의 마스킹을 이미 구현해 두었습니다. 이에 대한 자세한 내용은 아래에서 논의하겠습니다 (permanent hooks는 TransformerLens에 최근 추가된 기능으로 아직 다루지 않았으며, 이해해 두면 유용합니다).

### 기타 세부 사항

관련된 모든 아키텍처 세부 사항의 요약은 다음과 같습니다:

* Positional embedding은 sinusoidal(학습되지 않음) 방식입니다.
* `hidden_size` (또는 `d_model`, 또는 `embed_dim`)가 56입니다.
* BERT와 같이 bidirectional attention을 사용합니다.
* 3개의 attention layer와 3개의 MLP가 있습니다.
* 각 attention layer는 두 개의 head를 가지며, 각 head는 `headsize` (또는 `d_head`)가 `hidden_size / num_heads = 28`입니다.
* MLP hidden layer는 56개의 neuron을 가집니다 (즉, linear layer들이 정사각 행렬입니다).
* GPT와 마찬가지로, 각 attention layer와 각 MLP의 입력은 먼저 layernorm을 거칩니다.
* 모든 attention layer와 MLP가 더해진 후 residual stream에 LayerNorm이 적용됩니다 (이 또한 GPT와 유사합니다).
* embedding matrix `W_E`은 5개의 행을 가지며, 각각 token `[start]`, `[pad]`, `[end]`, `(`, `)` 순서대로 대응됩니다.
* unembedding matrix `W_U`는 2개의 열을 가지며, 각각 class `unbalanced`와 `balanced` 순서대로 대응됩니다.
    * 모델을 실행하면 `[batch, seq_len, 2]` 형태의 출력을 얻게 되며, 이후 `[:, 0, :]` 슬라이스를 취해 `[start]` token에 대한 출력(즉, classification logit)을 얻습니다.
    * 그 다음 softmax를 적용하여 classification 확률을 얻을 수 있습니다.
* Activation function은 `ReLU`입니다.

attention head를 지칭할 때는 다시 `layer.head`라는 약칭을 사용하며, layer와 head 모두 zero-indexed입니다. 따라서 `2.1`은 세 번째 layer(인덱스 2)의 두 번째 attention head(인덱스 1)를 의미합니다.

### 유용한 다이어그램들

다음은 모델 아키텍처의 하이레벨 다이어그램입니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/bracket-transformer-entire-model-short.png" width="800">

다음은 [link to a diagram of the archicture of a single model layer](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/diagram-tl.png) (activation의 이름과 모델 인덱싱에 유용한 메서드 목록이 포함되어 있습니다) 입니다.

이 두 이미지를 각각 다른 탭에 띄워두시는 것을 권장합니다.

### 모델 정의하기

여기서는 위에서 설명한 내용에 따라 모델을 정의합니다.

In [ ]:
VOCAB = "()"

cfg = HookedTransformerConfig(
    n_ctx=42,
    d_model=56,
    d_head=28,
    n_heads=2,
    d_mlp=56,
    n_layers=3,
    attention_dir="bidirectional",  # defaults to "causal"
    act_fn="relu",
    d_vocab=len(VOCAB) + 3,  # plus 3 because of end and pad and start token
    d_vocab_out=2,  # 2 because we're doing binary classification
    use_attn_result=True,
    device=device,
    use_hook_tokens=True,
)

model = HookedTransformer(cfg).eval()

state_dict = t.load(section_dir / "brackets_model_state_dict.pt", map_location=device)
model.load_state_dict(state_dict)

## Tokenizer

우리의 vocabulary에는 `[start]`, `[pad]`, `[end]`, `(`, `)` 순서로 단 다섯 개의 token만 존재합니다. 이 token들이 무엇을 나타내는지 다시 확인하시려면 이전 섹션을 참조하시기 바랍니다.

몇 가지 기본 함수를 제공하는 tokenizer `SimpleTokenizer("()")` 가 제공되었습니다. 다음 코드를 실행하여 어떤 기능을 하는지 확인해 보십시오:

In [ ]:
tokenizer = SimpleTokenizer("()")

# Examples of tokenization
# (the second one applies padding, since the sequences are of different lengths)
print(tokenizer.tokenize("()"))
print(tokenizer.tokenize(["()", "()()"]))

# Dictionaries mapping indices to tokens and vice versa
print(tokenizer.i_to_t)
print(tokenizer.t_to_i)

# Examples of decoding (all padding tokens are removed)
print(tokenizer.decode(t.tensor([[0, 3, 4, 2, 1, 1]])))

### 마스킹 구현하기

이제 tokenizer가 준비되었으므로, 이를 사용하여 padding token을 마스킹하는 hook을 작성할 수 있습니다. padding이 어떻게 작동하는지 이해하고 있다면, 이 코드의 모든 구현 세부 사항을 완벽히 따라가지 못하더라도 걱정하지 마십시오.

<details>
<summary>이 마스킹이 어떻게 작동하는지 설명하는 다이어그램을 보려면 클릭하십시오 (아래 코드를 이해하는 데 도움이 될 것입니다)</summary>

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/masking-padding-tokens.png" width="840">

</details>

In [ ]:
def add_perma_hooks_to_mask_pad_tokens(model: HookedTransformer, pad_token: int) -> HookedTransformer:
    # Hook which operates on the tokens, and stores a mask where tokens equal [pad]
    def cache_padding_tokens_mask(tokens: Float[Tensor, "batch seq"], hook: HookPoint) -> None:
        hook.ctx["padding_tokens_mask"] = einops.rearrange(tokens == pad_token, "b sK -> b 1 1 sK")

    # Apply masking, by referencing the mask stored in the `hook_tokens` hook context
    def apply_padding_tokens_mask(
        attn_scores: Float[Tensor, "batch head seq_Q seq_K"],
        hook: HookPoint,
    ) -> None:
        attn_scores.masked_fill_(model.hook_dict["hook_tokens"].ctx["padding_tokens_mask"], -1e5)
        if hook.layer() == model.cfg.n_layers - 1:
            del model.hook_dict["hook_tokens"].ctx["padding_tokens_mask"]

    # Add these hooks as permanent hooks (i.e. they aren't removed after functions like run_with_hooks)
    for name, hook in model.hook_dict.items():
        if name == "hook_tokens":
            hook.add_perma_hook(cache_padding_tokens_mask)
        elif name.endswith("attn_scores"):
            hook.add_perma_hook(apply_padding_tokens_mask)

    return model


model.reset_hooks(including_permanent=True)
model = add_perma_hooks_to_mask_pad_tokens(model, tokenizer.PAD_TOKEN)

## 데이터셋

각 학습 예제는 `[start]`, 최대 40개의 괄호, `[end]`, 그리고 필요한 만큼의 `[pad]`로 구성됩니다.

우리가 사용하는 데이터셋에서는 시퀀스의 절반은 균형 잡혀 있고, 나머지 절반은 균형이 맞지 않습니다. 동일한 분포를 갖게 한 것은 모델이 더 쉽게 학습할 수 있도록 하기 위함입니다.

아직 다운로드하지 않으셨다면 [this Google Drive link](https://drive.google.com/drive/folders/18gAF9HuiW9NG0MP2Gq8M7VdhXoKKxymT)에서 `brackets_data.json` 파일을 다운로드하는 것을 잊지 마십시오.

In [ ]:
N_SAMPLES = 5000
with open(section_dir / "brackets_data.json") as f:
    data_tuples = json.load(f)
    print(f"loaded {len(data_tuples)} examples, using {N_SAMPLES}")
    data_tuples = data_tuples[:N_SAMPLES]

data = BracketsDataset(data_tuples).to(device)
data_mini = BracketsDataset(data_tuples[:100]).to(device)

`data` 객체가 어떤 메서드와 속성을 가지고 있는지 확인하기 위해 `BracketsDataset`의 코드를 살펴보시는 것을 권장합니다 (상단의 설정 코드로 스크롤을 올리시되, 정답 코드를 너무 자세히 보지 않도록 주의하십시오!).

#### 데이터 시각화

좋은 관행에 따라, 데이터셋을 살펴보고 시퀀스 길이의 분포를 (예: 히스토그램으로) 그려보겠습니다. 어떤 점이 눈에 띄나요?

In [ ]:
hist(
    [len(x) for x, _ in data_tuples],
    nbins=data.seq_length,
    title="Sequence lengths of brackets in dataset",
    labels={"x": "Seq len"},
)

<details>
<summary>데이터셋의 특징</summary>

가장 눈에 띄는 특징은 모든 괄호 문자열의 길이가 짝수라는 점입니다. 우리가 데이터셋을 이렇게 구성한 이유는, 만약 홀수 길이의 문자열이 포함되어 있었다면 모델이 "문자열 길이가 홀수이면 불균형하다"라는 휴리스틱을 학습했을 가능성이 크기 때문입니다. 이는 학습하기 매우 쉬운 내용이며, 우리는 단순히 길이를 확인하는 것이 아니라 transformer가 괄호 문자열의 구조를 어떻게 학습하는지에 대한 더 흥미로운 질문에 집중하고자 합니다.

**보너스 연습 문제 (선택 사항) - 모델이 짝수 길이와 홀수 길이의 괄호 문자열을 구분하기 위해 사용할 수 있는, 단일 attention head를 포함한 알고리즘을 설명할 수 있습니까?**

<details>
<summary>정답</summary>

알고리즘은 다음과 같을 수 있습니다:

- QK circuit이 head로 하여금 seqpos=0에서 마스킹되지 않은 가장 마지막 sequence position을 attend하게 합니다 (예를 들어, positional embeddings `q[0] @ k[i]`의 key-query 내적이 `i = 0, 1, 2, ...`의 감소 함수가 되도록 설정할 수 있습니다).
- OV circuit이 positional embeddings의 parity 성분을 예측값으로 매핑합니다. 즉, 모든 홀수 위치는 "unbalanced" 예측으로, 짝수 위치는 "balanced" 예측으로 매핑됩니다.

추가 연습 문제로, 이러한 head를 직접 설계해 볼 수 있습니까?

</details>

</details>

이제 모든 준비가 되었으므로, 데이터에 모델을 실행하여 몇 가지 예측을 생성해 보겠습니다.

In [ ]:
# Define and tokenize examples
examples = ["()()", "(())", "))((", "()", "((()()()()))", "(()()()(()(())()", "()(()(((())())()))"]
labels = [True, True, False, True, True, False, True]
toks = tokenizer.tokenize(examples)

# Get output logits for the 0th sequence position (i.e. the [start] token)
logits = model(toks)[:, 0]

# Get the probabilities via softmax, then get the balanced probability (which is the second element)
prob_balanced = logits.softmax(-1)[:, 1]

# Display output
print(
    "Model confidence:\n"
    + "\n".join(
        [f"{ex:18} : {prob:<8.4%} : label={int(label)}" for ex, prob, label in zip(examples, prob_balanced, labels)]
    )
)

전체 데이터셋에 대해 모델을 실행하여 얼마나 많은 괄호가 올바르게 분류되는지 확인할 수도 있습니다.

In [ ]:
def run_model_on_data(
    model: HookedTransformer, data: BracketsDataset, batch_size: int = 200
) -> Float[Tensor, "batch 2"]:
    """Return probability that each example is balanced"""
    all_logits = []
    for i in tqdm(range(0, len(data.strs), batch_size)):
        toks = data.toks[i : i + batch_size]
        logits = model(toks)[:, 0]
        all_logits.append(logits)
    all_logits = t.cat(all_logits)
    assert all_logits.shape == (len(data), 2)
    return all_logits


test_set = data
n_correct = (run_model_on_data(model, test_set).argmax(-1).bool() == test_set.isbal).sum()
print(f"\nModel got {n_correct} out of {len(data)} training examples correct!")

## 알고리즘적 해결책

### 연습 문제 - 수기 솔루션 (for 루프)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than ~10 minutes on this exercise.
> This exercise and the next one should both be relatively easy (especially if you've already solved this problem on LeetCode before!), and they're very important for the rest of the exercises.
> ```

이렇게 간단한 문제를 사용하면 얻을 수 있는 좋은 점은 정답 솔루션을 직접 작성할 수 있다는 것입니다. for 루프와 if 문을 사용하여 이를 구현해 보시기 바랍니다.

In [ ]:
def is_balanced_forloop(parens: str) -> bool:
    """
    Return True if the parens are balanced.

    Parens is just the ( and ) characters, no begin or end tokens.
    """
    raise NotImplementedError()


for parens, expected in zip(examples, labels):
    actual = is_balanced_forloop(parens)
    assert expected == actual, f"{parens}: expected {expected} got {actual}"

print("All tests for `is_balanced_forloop` passed!")

<details><summary>솔루션</summary>

```python
def is_balanced_forloop(parens: str) -> bool:
    """
    Return True if the parens are balanced.

    Parens is just the ( and ) characters, no begin or end tokens.
    """
    cumsum = 0
    for paren in parens:
        cumsum += 1 if paren == "(" else -1
        if cumsum < 0:
            return False

    return cumsum == 0
```
</details>

### 연습 문제 - 손글씨 풀이 (벡터화)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than ~10 minutes on this exercise.
> ```

transformer는 각 시퀀스 위치에서 동일한 가중치가 서로 다른 데이터에 대해 "실행"되기 때문에, 벡터화된 연산에 대한 **inductive bias**를 가지고 있습니다. 따라서 우리가 "transformer처럼 생각"하고자 한다면, 절차적인 for/if 문에서 벗어나 적은 수의 transformer 가중치로 어떤 종류의 솔루션을 표현할 수 있을지 고민해야 합니다.

솔루션을 행렬 가중치로 표현할 수 있다는 점은 필요조건이지만, transformer가 일부 입력 데이터에 대해 SGD를 실행하여 해당 솔루션을 학습할 수 있음을 보여주기에는 충분하지 않습니다. 단순한 솔루션이 존재하더라도, 무작위 초기화 상태에서 현재의 optimizer 알고리즘을 사용할 때는 다른 솔루션이 attractor가 되는 경우가 있을 수 있기 때문입니다.

In [ ]:
def is_balanced_vectorized(tokens: Float[Tensor, "seq_len"]) -> bool:
    """
    Return True if the parens are balanced.

    tokens is a vector which has start/pad/end indices (0/1/2) as well as left/right brackets (3/4)
    """
    raise NotImplementedError()


for tokens, expected in zip(tokenizer.tokenize(examples), labels):
    actual = is_balanced_vectorized(tokens)
    assert expected == actual, f"{tokens}: expected {expected} got {actual}"

print("All tests for `is_balanced_vectorized` passed!")

<details>
<summary>힌트</summary>

```python
One solution is to map begin, pad, and end tokens to zero, map open paren to 1 and close paren to -1. Then take the cumulative sum, and check the two conditions which are necessary and sufficient for the bracket string to be balanced.
```

</details>


<details><summary>솔루션</summary>

```python
def is_balanced_vectorized(tokens: Float[Tensor, "seq_len"]) -> bool:
    """
    Return True if the parens are balanced.

    tokens is a vector which has start/pad/end indices (0/1/2) as well as left/right brackets (3/4)
    """
    # Convert start/end/padding tokens to zero, and left/right brackets to +1/-1
    table = t.tensor([0, 0, 0, 1, -1])
    change = table[tokens]
    # Get altitude by taking cumulative sum
    altitude = t.cumsum(change, -1)
    # Check that the total elevation is zero and that there are no negative altitudes
    no_total_elevation_failure = altitude[-1] == 0
    no_negative_failure = altitude.min() >= 0

    return (no_total_elevation_failure & no_negative_failure).item()
```
</details>

## 모델의 해결 방식

모델은 다음과 같은 방식으로 문제를 해결하는 것으로 나타났습니다:

각 위치 `i`에서, 모델은 현재 위치부터 끝까지 이어지는 슬라이스를 확인합니다: `seq[i:]`. 그런 다음 해당 슬라이스에 대해 (닫는 괄호 개수 빼기 여는 괄호 개수)를 계산하여 해당 위치의 출력을 생성합니다.

우리는 이 출력을 `i`에서의 "elevation" 또는 동일하게 각 suffix `seq[i:]`에 대한 elevation이라고 부르겠습니다.

다음 중 하나 또는 둘 다 참이면 시퀀스는 불균형한 상태입니다:

- `elevation[0]`가 0이 아님
- `any(elevation < 0)`

영어 사용자에게는 시퀀스를 왼쪽에서 오른쪽으로 처리하고 suffix 대신 prefix 슬라이스 `seq[:i]`에 대해 생각하는 것이 자연스럽지만, 모델은 bidirectional하며 영어가 무엇인지 알지 못합니다. 이 모델은 우연히 오른쪽에서 왼쪽으로 진행하는 동일하게 유효한 해결 방법을 학습했습니다.

우리는 오늘 네트워크의 서로 다른 부분들을 조사하여 다양한 layer들이 이 알고리즘을 어떻게 구현하는지 일차적으로 이해해 보겠습니다. 하지만 간단한 작업을 위해 학습된 모델이라 하더라도 neural network는 복잡하며, 우리는 퍼즐 조각의 아주 일부만을 탐색할 수 있을 것입니다.

# 2️⃣ 역방향으로 추적하기

> ##### 학습 목표
>
> * logit attribution을 수행하는 방법을 이해합니다.
> * 모델을 역방향으로 추적하여 최종 classification 확률에 어떤 컴포넌트가 가장 중요한지 식별하는 방법을 이해합니다.
> * LayerNorm이 어떻게 작동하는지 이해하고, 모델에서 이를 처리하는 몇 가지 방법을 살펴봅니다.

여기에서 여러분은 logit attribution을 수행하며, 모델의 특정 경로를 역추적하여 최종 classification 확률에 어떤 컴포넌트가 가장 중요한 영향을 미치는지 알아내는 방법을 배웁니다. 이번 단계는 모델에서 **LayerNorm**을 처음으로 다루게 되는 과정입니다.

*induction heads에 대해 logit attribution을 수행해 본 적이 있다면 이 섹션이 익숙하실 것입니다 (다만, 이번 실습은 코딩 관점에서 약간 더 까다롭습니다). LayerNorm 기반의 실습은 다소 세밀한 작업이 필요합니다!*

---

모델을 특정 시퀀스에 대해 실행하여 분류 확률 `[0.99, 0.01]`을 출력하고, "unbalanced"라고 매우 확신하며 분류했다고 가정해 보겠습니다.

우리는 모델이 왜 이러한 출력을 냈는지 알고 싶으며, 이를 위해 네트워크를 역방향으로 추적하며 초기 activation에 관한 사실과 최종 출력에 관한 사실 사이의 대응 관계를 파악할 것입니다. 모델의 계산 그래프 내 서로 다른 지점들을 연결하는 연결 고리를 구축하여, 나중 값에 대한 질문을 이전 값에 대한 질문으로 반복적으로 환원하고자 합니다.

쉬운 예시부터 시작하겠습니다. softmax는 상수를 더해도 값이 변하지 않으므로, 최종 분류 확률은 오직 class logit 간의 차이에만 의존한다는 점에 주목하십시오. 따라서 "무엇이 balanced에 대한 이 확률을 만들었는가?"라고 묻는 대신, 동일하게 "무엇이 이러한 logit 차이를 만들었는가?"라고 물을 수 있습니다. 한 단계 더 역방향으로 이동해 보겠습니다. logit은 각각 최종 LayerNorm 출력의 선형 함수이므로, 그 차이 또한 어떤 선형 함수가 될 것입니다. 다시 말해, LayerNorm 출력과 해당 벡터의 내적이 logit 차이가 되도록 하는 LayerNorm 출력 공간 상의 벡터를 찾을 수 있습니다.

이제 모델의 어느 부분이 의미 있는 역할을 하고 있는지 판단할 방법이 필요합니다. 우리는 start token의 embedding 공간에서 "unbalanced 방향", 즉 입력 문자열이 unbalanced임을 가장 잘 나타내는 단일 방향을 식별함으로써 이를 수행할 것입니다. 다른 방향들 또한 중요할 수 있다는 점(특히 layer norm 때문에)을 유의하는 것이 중요하지만, 첫 번째 근사치로서는 이 방법이 잘 작동합니다.

우리는 모델 출력부터 시작하여 역방향으로 작업하며, 각 단계에서 unbalanced 방향을 찾아낼 것입니다.

## residual stream으로 돌아가기

모델의 마지막 부분은 classification head이며, 이는 최종 layernorm, unembedding, 그리고 softmax의 세 단계로 구성됩니다. 이 과정의 끝에서 우리는 확률 값을 얻게 됩니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/bracket-transformer-first-attr-0.png" width="500">

참고 - 단순화를 위해, 다음 논의에서는 batch 차원을 무시하겠습니다.

다이어그램에 있는 객체들의 shape에 관한 몇 가지 참고 사항입니다:

* `x_2`은 layer 2의 attention heads와 MLPs를 거친 후의 residual stream에 있는 벡터입니다. 이 벡터의 shape은 `(seq_len, d_model)` 입니다.
* `final_ln_output`의 shape은 `(seq_len, d_model)` 입니다.
* `W_U`의 shape은 `(d_model, 2)` 이며, 따라서 `logits`의 shape은 `(seq_len, 2)` 입니다.
* 시퀀스 위치 0에 대해, softmax를 거친 logits의 0번째 요소를 가져옴으로써 `P(unbalanced)`를 얻습니다.

### Stage 1: softmax를 통한 변환

`P(unbalanced)`을 logit의 함수로 나타내 보겠습니다. 다행히 이는 간단합니다. 두 요소에 대해 softmax를 수행하므로, 두 logit 차이의 sigmoid로 단순화됩니다:

$$
\text{softmax}\left(\begin{bmatrix} \text{logit}_0 \\ \text{logit}_1 \end{bmatrix}\right)_0 = \frac{e^{\text{logit}_0}}{e^{\text{logit}_0} + e^{\text{logit}_1}} = \frac{1}{1 + e^{\text{logit}_1 - \text{logit}_0}} = \text{sigmoid}(\text{logit}_0 - \text{logit}_1)
$$

sigmoid는 단조 함수이므로, logit의 $\text{logit}_0 - \text{logit}_1$이 클수록 $\hat{y}_0$의 값도 커집니다. 이제부터는 "무엇이 logit의 큰 차이를 만드는가?"라는 질문에만 집중하겠습니다.

### 단계 2: linear를 통한 변환

다음으로 마주하게 될 단계는 decoder인 `logits = final_LN_output @ W_U`이며, 여기서

* `W_U`의 shape은 `(d_model, 2)`입니다
* `final_LN_output`의 shape은 `(seq_len, d_model)`입니다

이제 logit의 차이를 $W$와 $x_{\text{linear}}$의 함수로 다음과 같이 나타낼 수 있습니다:

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">logit_diff = (final_LN_output @ W_U)[0, 0] - (final_LN_output @ W_U)[0, 1]
           = final_LN_output[0, :] @ (W_U[:, 0] - W_U[:, 1])</pre>

(행렬 `AB`의 `(i, j)`번째 요소가 `A[i, :] @ B[:, j]`임을 상기하십시오)

따라서 logit의 큰 차이는 LayerNorm 출력값과 그에 대응하는 unembedding 벡터의 높은 내적(dot product)에서 비롯됩니다. 우리는 이를 `post_final_ln_dir`, 즉 최종 layernorm *이후* residual stream 값들에 대한 **unbalanced direction**이라고 부르겠습니다.

### 연습 문제 - `post_final_ln_dir` 가져오기

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than ~5 minutes on this exercise.
> ```

아래 함수에서 이 벡터를 계산해야 합니다 (이 함수는 단 한 줄로 작성되어야 합니다).

In [ ]:
def get_post_final_ln_dir(model: HookedTransformer) -> Float[Tensor, "d_model"]:
    """
    Returns the direction in which final_ln_output[0, :] should point to maximize P(unbalanced)
    """
    raise NotImplementedError()


tests.test_get_post_final_ln_dir(get_post_final_ln_dir, model)

<details><summary>솔루션</summary>

```python
def get_post_final_ln_dir(model: HookedTransformer) -> Float[Tensor, "d_model"]:
    """
    Returns the direction in which final_ln_output[0, :] should point to maximize P(unbalanced)
    """
    return model.W_U[:, 0] - model.W_U[:, 1]
```
</details>

### Stage 3: LayerNorm을 통한 변환

우리는 최종 layer norm 이전의 unbalanced direction을 찾고자 합니다. 왜냐하면 이곳이 residual stream을 항들의 합으로 표현할 수 있는 지점이기 때문입니다. LayerNorm은 비선형적이기 때문에 이러한 방향 분석을 어렵게 만듭니다. 하지만 오늘은 이를 선형 근사(linear fit)로 대체하겠습니다. 이는 흥미로운 분석을 수행하기에 충분한 수준입니다 ($R^2$ 값이 근사치에 대해 매우 높다는 것을 직접 확인해 보십시오)!

LayerNorm에 대한 선형 근사(여기서는 행렬 `L_final`을 사용합니다)를 통해, "LayerNorm 출력과 unbalanced-vector의 dot product는 무엇인가?"라는 질문을 LN 입력에 대한 질문으로 변환할 수 있습니다. 다음과 같이 간단히 쓸 수 있습니다:

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">final_ln_output[0, :] = final_ln(x_linear[0, :])
                      = L_final @ x_linear[0, :]</pre>

<details>
<summary>layernorm에 관한 부연 설명</summary>

Layernorm은 실제로 선형이 아닙니다. 이는 비선형 함수(평균을 빼고 표준편차로 나누는 과정)와 선형 함수(학습된 affine transformation)의 조합입니다.

하지만 이 경우에는 선형 근사를 사용하는 것이 꽤 괜찮은 근사치임이 밝혀졌습니다. 이 연습 문제에 layernorm을 포함시킨 이유는 비선형 함수가 우리의 분석을 어떻게 복잡하게 만들 수 있는지, 그리고 이를 처리하기 위한 몇 가지 간단하고 임시적인 방법들을 알려드리기 위해서입니다.

이러한 분석을 LLM에 적용할 때, layernorm을 단순히 선형 변환으로 추상화하는 것이 더 어려울 때가 있습니다. 예를 들어, 많은 대형 transformer는 residual stream의 일부를 "제거"하기 위해 layernorm을 사용합니다. 즉, 다른 모든 것보다 100배 더 큰 feature를 학습하고, layer norm을 사용하여 해당 요소를 제외한 residual stream의 모든 내용을 지워버리는 식입니다. 분명히 이러한 동작은 선형 근사로 잘 모델링되지 않습니다.

</details>

### 요약

우리는 logit diff를 모델이 괄호 문자열을 얼마나 강하게 unbalanced로 분류하고 있는지에 대한 척도로 사용할 수 있습니다 (logit diff가 높을수록 문자열이 unbalanced라고 더 확신하는 것입니다).

우리는 logit diff를 `pre_final_ln_dir`의 선형 함수로 근사할 수 있습니다 (unembedding이 선형이고, layernorm이 대략적으로 선형이기 때문입니다). 이는 logit diff를 최종 layernorm 이전의 residual stream 값과 `post_final_ln_dir`의 **dot product**로 근사할 수 있음을 의미합니다. 만약 이 `post_final_ln_dir`을 찾을 수 있다면, 어떤 컴포넌트의 출력이 이 값과 가장 높은 dot product를 가졌는지와 같은 다른 질문들에 답하기 시작할 수 있습니다.

아래 다이어그램은 모델을 역추적하여 **unbalanced direction** `pre_final_ln_dir`을 찾는 방법을 보여줍니다. 표기법: $x_2$는 layer 2의 attention heads와 MLP 이후의 residual stream 값(즉, 마지막 layernorm 직전)을 나타내며, $L_{final}$은 최종 layernorm의 선형 근사치입니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/brackets-untitled.png" width="1100">

### 연습 문제 - `pre_final_ln_dir` 구하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than 20-30 minutes on the following exercises.
> ```

이상적으로는 `post_final_ln_dir`에서 했던 것처럼 모델의 weight로부터 `pre_final_ln_dir`를 직접 계산하는 것이 좋습니다. 하지만 이번 경우에는 그렇게 간단하지 않습니다. 왜냐하면 선형 근사치인 `L_final`를 얻기 위해서는 모델을 통과하는 실제 데이터를 사용하여 선형 회귀(linear regression)를 적합시켜야 하기 때문입니다.

아래에서, 모델의 layernorm 중 하나의 입력과 출력에 대해 선형 회귀를 적합시키는 함수 `get_ln_fit`와, (위 다이어그램에 표시된 대로) `pre_final_ln_dir`의 값을 추정하는 `get_pre_final_ln_dir`을 구현해야 합니다.

몇 가지 헬퍼 함수를 제공해 드립니다:

- `run_with_cache` 함수를 사용하여 주어진 token batch에 대해 하나 또는 여러 개의 activation을 반환하는 `get_activation(s)`
- 모델 내의 layernorm(예: `model.ln_final`)을 입력받아 해당 layernorm 직전 또는 직후의 hook 이름을 반환하는 `LN_hook_names`. 이는 `get_activation(s)` 함수에서 이러한 값들을 참조할 때 유용합니다 (선형 회귀가 모델 layernorm의 입력과 출력에 대해 적합될 것이기 때문입니다).

회귀 적합을 수행할 때는 [sklearn LinearRegression class](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)을 사용하여 모델 layernorm의 입력과 출력에 대한 선형 적합을 찾는 것을 권장합니다. 회귀 분석에 적합 계수(fit coefficient)를 포함해야 합니다 (이는 `LinearRegression`의 기본 설정입니다).

참고로, 때로는 모든 sequence position에 대해 회귀를 적합시키고 싶을 때가 있고, 때로는 일부 position에만 관심이 있을 수 있기 때문에 `seq_pos` 인자를 추가했습니다 (예를 들어, 모델의 마지막 layernorm의 경우 예측값을 가져오는 0번째 position에만 관심이 있으며, 나머지 모든 position은 버려집니다).

In [ ]:
def get_activations(model: HookedTransformer, toks: Int[Tensor, "batch seq"], names: list[str]) -> ActivationCache:
    """Uses hooks to return activations from the model, in the form of an ActivationCache."""
    names_list = [names] if isinstance(names, str) else names
    _, cache = model.run_with_cache(
        toks,
        return_type=None,
        names_filter=lambda name: name in names_list,
    )
    return cache


def get_activation(model: HookedTransformer, toks: Int[Tensor, "batch seq"], name: str):
    """Gets a single activation."""
    return get_activations(model, toks, [name])[name]


def LN_hook_names(layernorm: nn.Module) -> tuple[str, str]:
    """
    Returns the names of the hooks immediately before and after a given layernorm.

    Example:
        model.final_ln -> ("blocks.2.hook_resid_post", "ln_final.hook_normalized")
    """
    if layernorm.name == "ln_final":
        input_hook_name = utils.get_act_name("resid_post", 2)
        output_hook_name = "ln_final.hook_normalized"
    else:
        layer, ln = layernorm.name.split(".")[1:]
        input_hook_name = utils.get_act_name("resid_pre" if ln == "ln1" else "resid_mid", layer)
        output_hook_name = utils.get_act_name("normalized", layer, ln)

    return input_hook_name, output_hook_name


def get_ln_fit(
    model: HookedTransformer,
    data: BracketsDataset,
    layernorm: nn.Module,
    seq_pos: int | None = None,
) -> tuple[LinearRegression, float]:
    """
    Fits a linear regression, where the inputs are the values just before the layernorm given by the
    input argument `layernorm`, and the values to predict are the layernorm's outputs.

    if `seq_pos` is None, find best fit aggregated over all sequence positions. Otherwise, fit only
    for the activations at `seq_pos`.

    Returns: A tuple of a (fitted) sklearn LinearRegression object and the r^2 of the fit.
    """
    raise NotImplementedError()


tests.test_get_ln_fit(get_ln_fit, model, data_mini)

_, r2 = get_ln_fit(model, data, layernorm=model.ln_final, seq_pos=0)
print(f"r^2 for LN_final, at sequence position 0: {r2:.4f}")
_, r2 = get_ln_fit(model, data, layernorm=model.blocks[1].ln1, seq_pos=None)
print(f"r^2 for LN1, layer 1, over all sequence positions: {r2:.4f}")

<details>
<summary>도움말 - linear regression을 어떻게 fit해야 할지 모르겠습니다.</summary>

만약 `inputs`와 `outputs`가 모두 `(samples, d_model)` shape의 tensor라면, `LinearRegression().fit(inputs, outputs)`은 fit object를 반환하며, 이것이 함수의 첫 번째 출력값이 되어야 합니다.

fit object의 `.score` 메서드를 통해 Rsquared 값을 얻을 수 있습니다.
</details>

<details>
<summary>도움말 - 서로 다른 <code>seq_pos</code> 케이스들을 어떻게 처리해야 할지 모르겠습니다.</summary>

만약 `seq_pos`가 정수라면, 해당 sequence position에 해당하는 벡터들만 가져와야 합니다. 다시 말해, `[batch, seq_pos, d_model]`-size tensor의 `[:, seq_pos, :]` slice를 가져와야 합니다.

만약 `seq_pos = None`라면, 모든 sequence position에 대해 한 번에 regression을 실행해야 하므로 tensor를 `(batch seq_pos) d_model`로 rearrange해야 합니다.
</details>


<details><summary>정답</summary>

```python
def get_ln_fit(
    model: HookedTransformer,
    data: BracketsDataset,
    layernorm: nn.Module,
    seq_pos: int | None = None,
) -> tuple[LinearRegression, float]:
    """
    Fits a linear regression, where the inputs are the values just before the layernorm given by the
    input argument `layernorm`, and the values to predict are the layernorm's outputs.

    if `seq_pos` is None, find best fit aggregated over all sequence positions. Otherwise, fit only
    for the activations at `seq_pos`.

    Returns: A tuple of a (fitted) sklearn LinearRegression object and the r^2 of the fit.
    """
    input_hook_name, output_hook_name = LN_hook_names(layernorm)

    activations_dict = get_activations(model, data.toks, [input_hook_name, output_hook_name])
    inputs = utils.to_numpy(activations_dict[input_hook_name])
    outputs = utils.to_numpy(activations_dict[output_hook_name])

    if seq_pos is None:
        inputs = einops.rearrange(inputs, "batch seq d_model -> (batch seq) d_model")
        outputs = einops.rearrange(outputs, "batch seq d_model -> (batch seq) d_model")
    else:
        inputs = inputs[:, seq_pos, :]
        outputs = outputs[:, seq_pos, :]

    final_ln_fit = LinearRegression().fit(inputs, outputs)

    r2 = final_ln_fit.score(inputs, outputs)

    return (final_ln_fit, r2)
```
</details>

#### 3. `pre_final_ln_dir` 계산하기

선형 회귀(linear fit) 결과를 바탕으로, 이제 마지막 layer norm 이전의 residual stream에서 불균형한 증거(unbalanced evidence) 방향을 가장 잘 가리키는 방향을 식별할 수 있습니다.

In [ ]:
def get_pre_final_ln_dir(model: HookedTransformer, data: BracketsDataset) -> Float[Tensor, "d_model"]:
    """
    Returns the direction in residual stream (pre ln_final, at sequence position 0) which
    most points in the direction of making an unbalanced classification.
    """
    raise NotImplementedError()


tests.test_get_pre_final_ln_dir(get_pre_final_ln_dir, model, data_mini)

<details><summary>솔루션</summary>

```python
def get_pre_final_ln_dir(model: HookedTransformer, data: BracketsDataset) -> Float[Tensor, "d_model"]:
    """
    Returns the direction in residual stream (pre ln_final, at sequence position 0) which
    most points in the direction of making an unbalanced classification.
    """
    post_final_ln_dir = get_post_final_ln_dir(model)

    final_ln_fit = get_ln_fit(model, data, layernorm=model.ln_final, seq_pos=0)[0]
    final_ln_coefs = t.from_numpy(final_ln_fit.coef_).to(device)

    return final_ln_coefs.T @ post_final_ln_dir
```
</details>

## residual stream을 항들의 합으로 표현하기

이전 연습 문제에서 보았듯이, residual stream을 모델을 통과하는 서로 다른 경로를 나타내는 항들의 합으로 생각하는 것이 훨씬 더 자연스럽습니다. 여기에는 residual stream에 값을 쓰는 10개의 컴포넌트가 있습니다: 직접 경로(즉, embedding), 그리고 3개의 각 레이어에 있는 2개의 attention head와 1개의 MLP입니다. 우리는 residual stream을 이러한 항들의 합으로 쓸 수 있습니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/attribution.png" width="900">

이렇게 하면, 분류에 직접적으로 기여하는 컴포넌트, 즉 balanced brackets에 비해 unbalanced brackets에 대한 `pre_final_ln_dir` 와 높은 내적(dot product)을 갖는 벡터를 residual stream에 쓰는 컴포넌트를 좁혀서 찾아낼 수 있습니다.

이 질문에 답하기 위해 다음 도구들이 필요합니다:
- LN으로 들어오는 입력을 컴포넌트별로 분해하는 방법.
- 네트워크가 'unbalanced'를 출력하게 만드는 embedding 공간의 방향을 식별하는 도구 (이미 가지고 있습니다).

### 연습 문제 - 구성 요소별로 residual stream 분해하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You shouldn't spend more than 15-20 minutes on this exercise.
> It isn't very conceptually important; the hardest part is getting all the right activation names & rearranging / stacking the tensors in the correct way.
> ```

`get_activations` 함수를 사용하여 구성 요소의 수가 10개인 `[num_components, dataset_size, seq_pos]` 모양의 tensor를 생성하십시오.

이는 각 구성 요소가 최종 layer norm으로 보내는 입력의 항별 표현입니다 (각 head가 residual stream에 무언가를 쓰고, 이것이 결국 최종 layer norm으로 전달된다고 생각하면 됩니다). 함수 출력의 구성 요소 순서는 위 다이어그램에 표시된 순서와 동일해야 합니다 (즉, residual stream에 추가되는 시간 순서대로여야 합니다).

(이 합계에서 누락된 유일한 항은 각 attention layer의 `W_O`-bias입니다).

<details>
<summary>이 bias 항이 누락된 이유에 대한 부연 설명입니다.</summary>

대부분의 다른 라이브러리는 `W_O`을 `[num_heads * d_head, d_model]` 모양의 2D tensor로 저장합니다. 이 경우, 행렬 `W_O`를 적용할 때 head들에 대한 합산이 계산 과정에서 암시적으로 이루어집니다. 그 후 길이가 `d_model`인 벡터인 `b_O`을 더합니다.

TransformerLens는 각 head의 출력을 개별적으로 쉽게 계산할 수 있도록 `W_O`를 `[num_heads, d_head, d_model]` 모양의 3D tensor로 저장합니다. TransformerLens는 다른 라이브러리와 호환되도록 설계되었으므로, bias 또한 `d_model` 모양이어야 하며, 이는 bias 항을 더하기 전에 head들에 대해 합산을 수행해야 함을 의미합니다. 따라서 개별 head의 출력 항에는 bias 항이 포함되지 않습니다.

실제로 여기서는 bias 항이 균형 잡힌 괄호와 균형 잡히지 않은 괄호에 대해 동일하므로 중요하지 않습니다. attribution을 수행할 때, 각 구성 요소에 대해 우리는 **균형 잡힌 시퀀스 대 균형 잡히지 않은 시퀀스**에 대해 그들이 residual stream에 쓰는 벡터의 균형 잡히지 않은 방향의 성분만을 고려합니다. bias는 모든 입력에서 동일합니다.
</details>

In [ ]:
def get_out_by_components(
    model: HookedTransformer, data: BracketsDataset
) -> Float[Tensor, "component batch seq_pos emb"]:
    """
    Computes a tensor of shape [10, dataset_size, seq_pos, emb] representing the output of the
    model's components when run on the data.

    The first dimension is the stacked components, in the following order:
        [embeddings, head 0.0, head 0.1, mlp 0, head 1.0, head 1.1, mlp 1, head 2.0, head 2.1, mlp 2]
    """
    raise NotImplementedError()


tests.test_get_out_by_components(get_out_by_components, model, data_mini)

<details><summary>솔루션</summary>

```python
def get_out_by_components(
    model: HookedTransformer, data: BracketsDataset
) -> Float[Tensor, "component batch seq_pos emb"]:
    """
    Computes a tensor of shape [10, dataset_size, seq_pos, emb] representing the output of the
    model's components when run on the data.

    The first dimension is the stacked components, in the following order:
        [embeddings, head 0.0, head 0.1, mlp 0, head 1.0, head 1.1, mlp 1, head 2.0, head 2.1, mlp 2]
    """
    embedding_hook_names = ["hook_embed", "hook_pos_embed"]
    head_hook_names = [utils.get_act_name("result", layer) for layer in range(model.cfg.n_layers)]
    mlp_hook_names = [utils.get_act_name("mlp_out", layer) for layer in range(model.cfg.n_layers)]

    all_hook_names = embedding_hook_names + head_hook_names + mlp_hook_names
    activations = get_activations(model, data.toks, all_hook_names)

    out = (activations["hook_embed"] + activations["hook_pos_embed"]).unsqueeze(0)

    for head_hook_name, mlp_hook_name in zip(head_hook_names, mlp_hook_names):
        out = t.concat(
            [
                out,
                einops.rearrange(activations[head_hook_name], "batch seq heads emb -> heads batch seq emb"),
                activations[mlp_hook_name].unsqueeze(0),
            ]
        )

    return out
```
</details>

이제 각 컴포넌트의 출력과 output projection bias의 합이 최종 layer norm의 입력과 일치하는지 확인하여 함수를 테스트할 수 있습니다.

In [ ]:
biases = model.b_O.sum(0)
out_by_components = get_out_by_components(model, data)
summed_terms = out_by_components.sum(dim=0) + biases

final_ln_input_name, final_ln_output_name = LN_hook_names(model.ln_final)
final_ln_input = get_activation(model, data.toks, final_ln_input_name)

t.testing.assert_close(summed_terms, final_ln_input)
print("Tests passed!")

<details>
<summary>힌트</summary>

먼저 모든 activation 이름을 리스트로 가져오는 것부터 시작하십시오. attention head의 output에 대한 activation 이름을 가져오려면 `utils.get_act_name("result", layer)`이 필요하며, MLP의 output에 대한 activation 이름을 가져오려면 `utils.get_act_name("mlp_out", layer)`가 필요합니다.

이 작업을 완료하고 `get_activations` 함수를 실행했다면, 이제 reshaping과 stacking을 수행하는 일만 남았습니다. embedding과 mlp activation의 shape은 `(batch, seq_pos, d_model)`이 되며, attention activation의 shape은 `(batch, seq_pos, head_idx, d_model)`이 됩니다.
</details>

### 어떤 컴포넌트가 중요한가요?

모델의 output이 "unbalanced"가 되는 데 어떤 컴포넌트가 직접적으로 중요한지 파악하기 위해, 실제로 unbalanced인 입력에 대해 어떤 컴포넌트가 position-0 residual stream으로 unbalanced 방향과 더 높은 dot product를 가진 벡터를 output하는 경향이 있는지 확인할 수 있습니다.

그 아이디어는, 만약 어떤 컴포넌트가 unbalanced 입력을 올바르게 분류하는 데 중요하다면, unbalanced bracket 문자열이 입력되었을 때의 벡터 output이 balanced bracket 문자열이 입력되었을 때보다 unbalanced 방향으로 더 높은 dot product를 가질 것이라는 점입니다.

이 섹션에서는 각 컴포넌트의 dot product에 대한 히스토그램을 그려보겠습니다. 이를 통해 어떤 컴포넌트가 유의미한지 관찰할 수 있습니다.

예를 들어, 컴포넌트 중 하나가 다음과 같이 bimodal output을 생성한다고 가정해 보겠습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/exampleplot.png" width="650">

이는 balanced 입력에 비해 unbalanced bracket 입력을 unbalanced 방향(즉, 입력이 unbalanced로 분류되는 데 기여하는 방향)으로 더 강하게 밀어내고 있으므로, **이 컴포넌트가 모델의 output이 unbalanced가 되는 데 중요하다는 강력한 증거**가 됩니다.

### 연습 문제 - 각 컴포넌트에 대해 unbalanced direction에서의 출력 계산하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 10-15 minutes on this exercise.
> 
> It's very important to conceptually understand what object you are computing here. The actual computation is just a few lines of code involving indexing and einsums.
> ```

아래 코드 블록에서 `(10, batch)` 크기의 `out_by_component_in_unbalanced_dir`라는 텐서를 계산해야 합니다. 이 텐서의 `[i, j]`번째 요소는 데이터셋의 `j`번째 시퀀스에 대해, `i`번째 컴포넌트의 출력과 unbalanced direction의 내적(dot product) 값이어야 합니다.

balanced 샘플들에 대해 이 컴포넌트의 출력과 unbalanced direction의 내적 값의 **평균을 빼줌으로써** 이를 정규화해야 합니다. 이렇게 하면 balanced 샘플에 해당하는 히스토그램이 0에 중심을 맞추게 되어(위 그림과 같이), 해석이 더 쉬워집니다. 우리가 관심을 갖는 것은 오직 **unbalanced 샘플과 balanced 샘플 간의 내적 값의 차이**뿐이라는 점을 기억하십시오 (두 logit 모두에 상수를 더하는 것은 모델의 확률적 출력에 영향을 주지 않기 때문입니다).

이 히스토그램들을 그려주는 `hists_per_comp` 함수가 제공됩니다. 여러분이 해야 할 일은 `out_by_component_in_unbalanced_dir` 객체를 계산하여 해당 함수에 전달하는 것입니다.

In [ ]:
# YOUR CODE HERE - define the object `out_by_component_in_unbalanced_dir`

tests.test_out_by_component_in_unbalanced_dir(out_by_component_in_unbalanced_dir, model, data)

plotly_utils.hists_per_comp(out_by_component_in_unbalanced_dir, data, xaxis_range=[-10, 20])

<details>
<summary>힌트</summary>

다음 두 객체를 정의하는 것부터 시작하십시오:

* 시퀀스 위치 0에서의 컴포넌트별 출력, 즉 `(component, batch, d_model)` 모양의 tensor입니다.
* 길이가 `d_model`인 `pre_final_ln_dir` 벡터입니다.

그 다음, 적절한 dot product를 계산하여 magnitude를 생성하십시오.

모든 balanced sample에 대해 각 컴포넌트의 평균을 빼는 것을 잊지 마십시오 (boolean `data.isbal`을 인덱스로 사용할 수 있습니다).
</details>


<details><summary>솔루션</summary>

```python
# Get output by components, at sequence position 0 (which is used for classification)
out_by_components_seq0 = out_by_components[:, :, 0, :]  # [component=10 batch d_model]
# Get the unbalanced direction for tensors being fed into the final layernorm
pre_final_ln_dir = get_pre_final_ln_dir(model, data)  # [d_model]
# Get the size of the contributions for each component
out_by_component_in_unbalanced_dir = einops.einsum(
    out_by_components_seq0,
    pre_final_ln_dir,
    "comp batch d_model, d_model -> comp batch",
)
# Subtract the mean
out_by_component_in_unbalanced_dir -= out_by_component_in_unbalanced_dir[:, data.isbal].mean(dim=1).unsqueeze(1)
```
</details>

어떤 head가 가장 중요하다고 생각하시나요? 그리고 그 이유가 무엇일지 추측해 보시겠습니까?

<details>
<summary>답변</summary>

layer 2의 head들(즉, `2.0` 및 `2.1`)이 가장 중요한 것으로 보입니다. 그 이유는 닫히지 않은 괄호들이 닫힌 괄호들보다 훨씬 더 오른쪽으로 밀려나고 있기 때문입니다.

여기서 일종의 composition이 일어나고 있다고 추측할 수 있습니다. layer 0 head들의 출력은 사실상 단일 레이어 transformer처럼 작동하기 때문에 composition에 관여할 수 없습니다. 하지만 이후의 레이어들은 입력값이 embedding뿐만 아니라 이전 레이어의 출력값에서도 오기 때문에 composition에 참여할 수 있습니다. 이는 이들이 더 복잡한 계산을 수행할 수 있음을 의미합니다.
</details>

### 실패 유형별 head 영향도

이 히스토그램들은 어떤 head가 중요한지를 보여주었지만, 이 head들이 구체적으로 무엇을 하고 있는지는 알려주지 않습니다. 이에 대한 단서를 얻기 위해, layer 2에 있는 두 개의 head에 집중하여 다양한 유형의 입력에 대해 우리가 선택한 방향으로 얼마나 많이 write하는지 살펴보겠습니다. 특히, 'overall elevation' 테스트와 'nowhere negative' 테스트를 통과하는지 여부에 따라 입력을 분류할 수 있습니다.

또한 닫는 괄호로 시작하는 문장들은 무시하겠습니다. 이러한 문장들에 대해서는 동작 방식이 다소 다르기 때문입니다 (이들은 즉시 unbalanced로 분류될 수 있으므로, 더 복잡한 로직이 필요하지 않습니다).

### 연습 문제 - 실패 유형별 괄호 문자열 분류

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You shouldn't spend more than 15-20 minutes on this exercise.
> These exercises should be pretty straightforward; you'll be able to use much of your code from previous exercises.
> They are also quite fiddly, so you should look at the solutions if you are stuck.
> ```

시각화가 정상적으로 작동하도록 다음 객체들을 정의하십시오:

* **`negative_failure`**
  * 이는 `(N_SAMPLES,)` boolean 벡터이며, (오른쪽에서 왼쪽으로 읽을 때) elevation이 한 번이라도 음수로 떨어지는 시퀀스, 즉 닫히지 않은 열린 괄호가 있는 경우에 true가 됩니다.                                                         |
* **`total_elevation_failure`**
  * 이는 `(N_SAMPLES,)` boolean 벡터이며, 전체 elevation이 정확히 0이 아닌 시퀀스에 대해 true가 됩니다. 다시 말해, 열린 괄호와 닫힌 괄호의 개수가 맞지 않는 문장을 의미합니다.                                                            |
* **`h20_in_unbalanced_dir`**
  * 이는 `(N_SAMPLES,)` float 벡터이며, unbalanced 방향으로 position-0 residual stream에 기여하는 head 2.0의 값과 같습니다. 이 값은 _balanced 시퀀스_들에 대해 해당 stream에 기여하는 평균 unbalancedness 기여도를 빼서 정규화합니다. |
* **`h21_in_unbalanced_dir`**
  * 위와 동일하지만 head 2.1에 해당합니다. |

처음 두 항목의 경우, `is_balanced_vectorized` 코드를 다시 참고하는 것이 도움이 될 것입니다 (다만, 여기서는 **오른쪽에서 왼쪽으로** 읽고 있다는 점을 기억하십시오. 이로 인해 결과가 *변경될 것입니다*!).

마지막 두 항목은 `out_by_component_in_unbalanced_dir` tensor에서 직접 인덱싱하여 얻을 수 있습니다.

In [ ]:
def is_balanced_vectorized_return_both(
    toks: Int[Tensor, "batch seq"],
) -> tuple[Bool[Tensor, "batch"], Bool[Tensor, "batch"]]:
    raise NotImplementedError()


total_elevation_failure, negative_failure = is_balanced_vectorized_return_both(data.toks)

h20_in_unbalanced_dir = out_by_component_in_unbalanced_dir[7]
h21_in_unbalanced_dir = out_by_component_in_unbalanced_dir[8]

tests.test_total_elevation_and_negative_failures(data, total_elevation_failure, negative_failure)

<details><summary>솔루션</summary>

```python
def is_balanced_vectorized_return_both(
    toks: Int[Tensor, "batch seq"],
) -> tuple[Bool[Tensor, "batch"], Bool[Tensor, "batch"]]:
    table = t.tensor([0, 0, 0, 1, -1]).to(device)
    change = table[toks.to(device)].flip(-1)
    altitude = t.cumsum(change, -1)
    total_elevation_failure = altitude[:, -1] != 0
    negative_failure = altitude.max(-1).values > 0
    return total_elevation_failure, negative_failure
```
</details>

테스트를 통과했다면, 아래 코드를 실행하여 plot을 생성할 수 있습니다.

In [ ]:
failure_types_dict = {
    "both failures": negative_failure & total_elevation_failure,
    "just neg failure": negative_failure & ~total_elevation_failure,
    "just total elevation failure": ~negative_failure & total_elevation_failure,
    "balanced": ~negative_failure & ~total_elevation_failure,
}

plotly_utils.plot_failure_types_scatter(h20_in_unbalanced_dir, h21_in_unbalanced_dir, failure_types_dict, data)

그래프를 살펴보고 서로 다른 head들의 역할이 무엇인지 생각해보세요!

<details>
<summary>스스로 생각한 후에 읽어보세요</summary>

여기서 얻어갈 핵심은 2.0은 열린 괄호와 닫힌 괄호의 전체 개수를 확인하는 역할을 담당하고, 2.1은 elevation이 절대 음수가 되지 않도록 보장하는 역할을 담당한다는 점입니다.

여담으로, 실제 이야기는 그보다 조금 더 복잡합니다. 두 head 모두 종종 자신의 책임이 아닌 실패 사례를 포착하여 'unbalanced' 방향으로 출력하곤 합니다. 이는 사실 log-loss에 의해 유도된 결과입니다. 불균형한 시퀀스에 대해 '책임'이 있는 head만 그렇게 출력하는 것보다, 두 head가 만장일치로 'unbalanced'를 출력할 때 loss가 약간 더 낮아지기 때문입니다. layer 1의 head들이 이를 돕는 일부 로직을 수행하지만, 오늘은 다루지 않겠습니다.

이를 생각하는 한 가지 방법은, 각 head가 자신이 담당하는 실패 유형에 대해 매우 신뢰할 수 있도록 전문화되었으며, 때로는 다른 유형의 실패도 성공적으로 포착한다는 것입니다.
</details>

나머지 대부분의 실습에서는 head 2.0에 의해 구현된 전체 elevation circuit에 집중할 것입니다. head 2.0이 무엇을 하고 있는지에 대한 직관을 얻기 위한 추가적인 방법으로, head 2.0의 출력을 시퀀스에서 열린 괄호가 차지하는 전체 비율에 대해 그래프로 그려보겠습니다.

In [ ]:
plotly_utils.plot_contribution_vs_open_proportion(
    h20_in_unbalanced_dir,
    "Head 2.0 contribution vs proportion of open brackets '('",
    failure_types_dict,
    data,
)

이를 head 2.1과 비교해 볼 수도 있습니다:

In [ ]:
plotly_utils.plot_contribution_vs_open_proportion(
    h21_in_unbalanced_dir,
    "Head 2.1 contribution vs proportion of open brackets '('",
    failure_types_dict,
    data,
)

# 3️⃣ 전체 고도(total elevation) 회로 이해하기

> ##### 학습 목표
>
> * 독특한 attention 패턴을 사람이 이해할 수 있는 알고리즘과 연결하고, 모델 동작에 대해 추론하는 연습을 합니다.
> * MLP를 뉴런들의 집합으로 보는 방법을 이해합니다.
> * 전체 고도 회로의 전체적인 모습과 작동 방식에 대한 완전한 그림을 그려 나갑니다.

연습 문제의 가장 큰 섹션에서, 여러분은 서로 다른 head들의 attention pattern을 조사하고, 이를 인간이 이해할 수 있는 알고리즘(예: copying 또는 aggregation)을 수행하는 것으로 해석하게 됩니다. 여러분은 관찰 결과를 바탕으로, 특정 유형의 balanced brackets failure mode(왼쪽과 오른쪽 괄호의 개수가 일치하지 않는 경우)가 모델에 의해 어떻게 감지되는지에 대한 추론을 수행할 것입니다. 이번 단계는 모델 내의 **MLP**를 처음으로 다루게 되는 과정입니다.

*이 섹션은 코딩과 개념적 관점 모두에서 상당히 도전적입니다. 관찰과 intervention의 결과를 모델이 작동하는 방식에 대한 구체적인 가설과 연결해야 하기 때문입니다.*

## 책임 헤드의 Attention pattern

token 0의 query가 여는 괄호일 때, 2.0은 어떤 token들에 attention을 기울이고 있습니까? 여는 괄호로 시작하는 sequence에 집중한다는 점을 상기하십시오. 그렇지 않은 sequence는 즉시 제외될 수 있으므로, 더 복잡한 동작은 필요하지 않기 때문입니다.

### 연습 문제 - attention 확률 가져오기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You shouldn't spend more than 5-10 minutes on this exercise.
> This exercise just involves the `get_activations` helper func, and some indexing.
> ```

입력 배치에 대해 실행했을 때, 특정 head의 attention pattern을 추출하는 함수를 작성하십시오.

In [ ]:
def get_attn_probs(model: HookedTransformer, data: BracketsDataset, layer: int, head: int) -> Tensor:
    """
    Returns: (N_SAMPLES, max_seq_len, max_seq_len) tensor that sums to 1 over the last dimension.
    """
    raise NotImplementedError()


tests.test_get_attn_probs(get_attn_probs, model, data_mini)

<details><summary>솔루션</summary>

```python
def get_attn_probs(model: HookedTransformer, data: BracketsDataset, layer: int, head: int) -> Tensor:
    """
    Returns: (N_SAMPLES, max_seq_len, max_seq_len) tensor that sums to 1 over the last dimension.
    """
    return get_activation(model, data.toks, utils.get_act_name("pattern", layer))[:, head, :, :]
```
</details>

테스트를 통과했다면, 결과를 plot 할 수 있습니다:

In [ ]:
attn_probs_20 = get_attn_probs(model, data, 2, 0)  # [batch seqQ seqK]
attn_probs_20_open_query0 = attn_probs_20[data.starts_open].mean(0)[0]

bar(
    attn_probs_20_open_query0,
    title="Avg Attention Probabilities for query 0, first token '(', head 2.0",
    width=700,
    template="simple_white",
    labels={"x": "Sequence position", "y": "Attn prob"},
)

포지션 1에서는 약 0.5의 평균 attention이 나타나고, 다른 모든 token에서는 약 0의 평균이 나타나는 것을 확인하실 수 있습니다. 따라서 `2.0`은 단순히 residual stream 1에서 residual stream 0으로 정보를 이동시키고 있는 것입니다. 다시 말해, `2.0`은 residual stream 1을 (물론 `LayerNorm` 과정을 거친 후) 자신의 `W_OV` circuit를 통해 통과시키며, 이때 우리는 상수로 가정하는 일정량의 가중치를 곱합니다. 중요한 점은, 이는 **분류에 필요한 정보가 이 head에 도달하기 전에 이미 시퀀스 포지션 1에 저장되어 있어야 함**을 의미한다는 것입니다. 상황이 점점 흥미진진해집니다!

### 이 head 이전의 의미 있는 방향 식별하기

head 2.0에 의해 시퀀스 포지션 0으로 이동한 벡터가 단순히 `layernorm(x[1]) @ W_OV` (여기서 `x[1]`는 head 2.0 이전, 시퀀스 포지션 1에서의 residual stream 벡터입니다)라고 단순화한다면, 이전에 수행했던 것과 동일한 방식의 logit attribution을 수행할 수 있습니다. 최종 layernorm으로 들어가는 입력(시퀀스 포지션 0에서)을 10개 컴포넌트의 합으로 분해하여 "pre final layernorm unbalanced direction"에서의 기여도를 측정하는 대신, head 2.0으로 들어가는 입력(시퀀스 포지션 1에서)을 head 2.0 이전의 7개 컴포넌트의 합으로 분해하고, "pre head 2.0 unbalanced direction"에서의 기여도를 측정할 수 있습니다.

우리가 정확히 무엇을 하고 있는지 더 잘 설명하기 위해 주석이 달린 다이어그램을 준비했습니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/bracket_transformer-elevation-circuit-1.png" width="900">

### 연습 문제 - pre-head 2.0 unbalanced direction 계산하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than 15-20 minutes on these exercises.
> The second function should be conceptually similar to `get_pre_final_ln_dir` from earlier.
> ```

아래에서 여러분은 `pre_20_dir`을 계산하게 됩니다. 이는 시퀀스 위치 1에서 head 2.0으로 들어가는 입력에 대한 unbalanced direction입니다 (이 시퀀스 위치의 벡터들이 head `2.0`에 의해 위치 0으로 복사된 후, 예측에 사용된다는 사실에 기반합니다).

먼저, 특정 layer와 head의 OV matrix를 가져오는 함수 `get_WOV`을 구현합니다. 이는 `W_O`와 `W_V` matrix의 곱이라는 점을 기억하십시오. 그 다음, 이 함수를 사용하여 `get_pre_20_dir`을 작성합니다.

In [ ]:
def get_WOV(model: HookedTransformer, layer: int, head: int) -> Float[Tensor, "d_model d_model"]:
    """
    Returns the W_OV matrix for a particular layer and head.
    """
    raise NotImplementedError()


def get_pre_20_dir(model, data) -> Float[Tensor, "d_model"]:
    """
    Returns the direction propagated back through the OV matrix of 2.0 and then through the
    layernorm before the layer 2 attention heads.
    """
    raise NotImplementedError()


tests.test_get_pre_20_dir(get_pre_20_dir, model, data_mini)

<details><summary>솔루션</summary>

```python
def get_WOV(model: HookedTransformer, layer: int, head: int) -> Float[Tensor, "d_model d_model"]:
    """
    Returns the W_OV matrix for a particular layer and head.
    """
    return model.W_V[layer, head] @ model.W_O[layer, head]


def get_pre_20_dir(model, data) -> Float[Tensor, "d_model"]:
    """
    Returns the direction propagated back through the OV matrix of 2.0 and then through the
    layernorm before the layer 2 attention heads.
    """
    W_OV = get_WOV(model, 2, 0)

    layer2_ln_fit, r2 = get_ln_fit(model, data, layernorm=model.blocks[2].ln1, seq_pos=1)
    layer2_ln_coefs = t.from_numpy(layer2_ln_fit.coef_).to(device)

    pre_final_ln_dir = get_pre_final_ln_dir(model, data)

    return layer2_ln_coefs.T @ W_OV @ pre_final_ln_dir
```
</details>

### 연습 문제 - 컴포넌트 크기 계산하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You shouldn't spend more than 10-15 minutes on these exercises.
> This exercise should be somewhat similar to the last time you computed component magnitudes.
> ```

이제 `pre_20_dir`을(를) 얻었으므로, 이전에 나왔던 각 컴포넌트의 크기를 계산할 수 있습니다. 혼란스럽다면 위의 다이어그램을 다시 참고하시기 바랍니다. **균형 잡힌 입력을 위해 각 컴포넌트의 평균을 빼는 것을 잊지 마십시오.**

In [ ]:
# YOUR CODE HERE - define `out_by_component_in_pre_20_unbalanced_dir` (for all components before head 2.0)
pre_layer2_outputs_seqpos1 = out_by_components[:-3, :, 1, :]
out_by_component_in_pre_20_unbalanced_dir = einops.einsum(
    pre_layer2_outputs_seqpos1,
    get_pre_20_dir(model, data),
    "comp batch emb, emb -> comp batch",
)
out_by_component_in_pre_20_unbalanced_dir -= out_by_component_in_pre_20_unbalanced_dir[:, data.isbal].mean(-1, True)

tests.test_out_by_component_in_pre_20_unbalanced_dir(out_by_component_in_pre_20_unbalanced_dir, model, data)

plotly_utils.hists_per_comp(out_by_component_in_pre_20_unbalanced_dir, data, xaxis_range=(-5, 12))

무엇이 관찰됩니까?

<details>
<summary>주목할 점들</summary>

한 가지 분명한 점은 embeddings 그래프가 0의 출력을 보여준다는 것입니다. 다시 말해, classification에 아무런 영향을 주지 않습니다. 이는 이 경로의 입력이 단지 0번째 sequence position에 있는 embedding 벡터, 즉 모든 입력에 대해 동일한 `[START]` token의 embedding이기 때문입니다.

---

더 흥미로운 점은, `mlp0`와 특히 `mlp1`가 매우 중요하다는 것을 알 수 있다는 점입니다. 이는 타당합니다. mlp가 특히 잘 수행하는 작업 중 하나가 더 연속적인 feature('이 입력에서 열린 괄호 문자의 비율은 얼마인가?')를 날카로운 불연속적 feature('그 비율이 정확히 0.5인가?')로 변환하는 것이기 때문입니다.

예를 들어, 합계 $\operatorname{ReLU}(x-0.5) + \operatorname{ReLU}(0.5-x)$은 비선형 함수 $|x-0.5|$로 평가되며, 이는 $x=0.5$인 경우에만 0이 됩니다. 이는 우리 모델이 열린 괄호가 정확히 50%가 아닌 한 모든 bracket 문자열을 unbalanced로 분류할 수 있는 한 가지 방법입니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/relu2.png" width="600">

---

Head `1.1` 또한 어느 정도 중요성을 가지지만, 오늘은 이에 대해 자세히 파헤치지는 않겠습니다. 결과적으로 이 head가 수행하는 주요 작업 중 하나는 negative elevation failure가 발생했을 때의 정보를 이 전체 elevation branch에 통합하는 것입니다. 이를 통해 head들은 열린 괄호와 닫힌 괄호의 전체 개수만으로는 balanced가 될 수 있는 상황일지라도, 명백하게 unbalanced인 경우 prompt가 unbalanced하다는 것에 동의할 수 있게 됩니다.

</details>

`mlp0`과 `mlp1`이 무엇을 하고 있는지 더 자세히 살펴보기 위해, 전체 open-proportion의 함수로서 이들의 출력을 확인할 수 있습니다.

In [ ]:
plotly_utils.mlp_attribution_scatter(out_by_component_in_pre_20_unbalanced_dir, data, failure_types_dict)

### key-value pair로서의 MLP

transformer를 처음부터 구현했을 때, 우리는 MLP를 key-value pair로 생각할 수 있다는 점을 관찰했습니다. 이를 짧게 요약하면 다음과 같습니다:

> MLP의 출력을 $f(x^T W^{in})W^{out}$로 쓸 수 있으며, 여기서 $W^{in}$과 $W^{out}$는 MLP의 서로 다른 가중치(bias 제외)이고, $f$은 activation 함수이며, $x$는 residual stream의 벡터입니다. 이는 다음과 같이 다시 쓸 수 있습니다:
>
> $$
> f(x^T W^{in}) W^{out} = \sum_{i=1}^{d_{mlp}} f(x^T W^{in}_{[:, i]}) W^{out}_{[i, :]}
> $$
>
> 우리는 벡터 $W^{in}_{[:, i]}$을 **입력 방향(input directions)**으로, $W^{out}_{[i, :]}$을 **출력 방향(output directions)**으로 볼 수 있습니다. 입력 방향은 특정 텍스트 특징에 의해 **활성화(activated)**된다고 하며, 이들이 활성화되면 해당 출력 방향으로 벡터가 기록됩니다. 이는 attention 레이어의 key와 value 개념과 매우 유사하며, 그렇기 때문에 이 벡터들을 때때로 key와 value라고 부르기도 합니다 (예: 논문 [Transformer Feed-Forward Layers Are Key-Value Memories](https://arxiv.org/pdf/2012.14913.pdf) 참조).

bias를 포함한 이 공식의 전체 버전은 다음과 같습니다:

$$
MLP(x) = \sum_{i=1}^{d_{mlp}}f(x^T W^{in}_{[:, i]} + b^{in}_i) W^{out}_{[i,:]} + b^{out}
$$

이를 설명하는 다이어그램입니다 (bias 제외):

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/mlp-neurons-2.png" width="850">

### 연습 문제 - 뉴런별 출력값 가져오기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 25-35 minutes on these exercises.
> It's important to understand exactly what the MLP is doing, and how to work with it.
> ```

함수 `get_out_by_neuron`은 주어진 MLP의 뉴런별 출력값을 반환해야 합니다. 다시 말해, 출력값은 `[batch, seq, neurons, d_model]` 형태를 가지며, 여기서 `out[b, s, i]`은 벡터 $f(\vec x^T W^{in}_{[:,i]} + b^{in}_i)W^{out}_{[i,:]}$ 입니다 (그리고 `i`에 대해 합산하면 실제 MLP의 출력값이 됩니다). 여기서는 $b^{out}$을 무시하는데, 이는 특정 뉴런에 귀속될 수 없기 때문입니다.

이 출력값을 얻으면, 시퀀스 위치 1에서 head 2.0으로 들어오는 입력에 대해 _불균형한 방향(unbalanced direction)_으로 각 뉴런의 출력값을 계산하기 위해 `get_out_by_neuron_in_20_dir`을 사용할 수 있습니다. head 2.0이 주로 위치 1에서 위치 0으로 정보를 복사한다는 점을 관찰했기 때문에, 시퀀스 위치 1만 고려한다는 점에 유의하십시오. 이것이 `get_out_by_neuron` 함수에 `seq` 인자를 제공한 이유이며, 이를 통해 필요한 것보다 더 많은 정보를 저장할 필요가 없습니다.

In [ ]:
def get_out_by_neuron(
    model: HookedTransformer, data: BracketsDataset, layer: int, seq: int | None = None
) -> Float[Tensor, "batch *seq neuron d_model"]:
    """
    If seq=None, then out[batch, seq, i, :] = f(x[batch, seq].T @ W_in[:, i] + b_in[i]) @ W_out[i, :],
    i.e. the vector which is written to the residual stream by the ith neuron (where x is the input to
    the residual stream (i.e. shape (batch, seq, d_model)).

    If seq is not None, then out[batch, i, :] = f(x[batch, seq].T @ W_in[:, i]) @ W_out[i, :], i.e. we
    just look at the sequence position given by argument seq.

    (Note, using * in jaxtyping indicates an optional dimension)
    """
    raise NotImplementedError()


def get_out_by_neuron_in_20_dir(
    model: HookedTransformer, data: BracketsDataset, layer: int
) -> Float[Tensor, "batch neurons"]:
    """
    [b, s, i]th element is the contribution of the vector written by the ith neuron to the residual stream in the
    unbalanced direction (for the b-th element in the batch, and the s-th sequence position).

    In other words we need to take the vector produced by the `get_out_by_neuron` function, and project it onto the
    unbalanced direction for head 2.0 (at seq pos = 1).
    """
    raise NotImplementedError()


tests.test_get_out_by_neuron(get_out_by_neuron, model, data_mini)
tests.test_get_out_by_neuron_in_20_dir(get_out_by_neuron_in_20_dir, model, data_mini)

<details>
<summary>힌트</summary>

`get_out_by_neuron` 함수를 위해, $f(\vec x^T W^{in}_{[:,i]} + b^{in}_i)$와 $W^{out}_{[i,:]}$를 각각 정의한 다음 서로 곱하십시오. 전자는 `"post"`이라는 이름에 해당하는 activation이며, `get_activations` 함수를 사용하여 접근할 수 있습니다. 후자는 단순히 모델 가중치이며, `model.W_out`을 사용하여 접근할 수 있습니다.

또한, activation과 parameter의 차이점을 명심하십시오. $f(\vec x^T W^{in}_{[:,i]} + b^{in}_i)$는 activation이며, `batch` 및 `seq_len` 차원을 가집니다. $W^{out}_{[i,:]}$는 parameter이며, `batch` 또는 `seq_len` 차원이 없습니다.
</details>


<details><summary>솔루션</summary>

```python
def get_out_by_neuron(
    model: HookedTransformer, data: BracketsDataset, layer: int, seq: int | None = None
) -> Float[Tensor, "batch *seq neuron d_model"]:
    """
    If seq=None, then out[batch, seq, i, :] = f(x[batch, seq].T @ W_in[:, i] + b_in[i]) @ W_out[i, :],
    i.e. the vector which is written to the residual stream by the ith neuron (where x is the input to
    the residual stream (i.e. shape (batch, seq, d_model)).

    If seq is not None, then out[batch, i, :] = f(x[batch, seq].T @ W_in[:, i]) @ W_out[i, :], i.e. we
    just look at the sequence position given by argument seq.

    (Note, using * in jaxtyping indicates an optional dimension)
    """
    # Get the W_out matrix for this MLP
    W_out = model.W_out[layer]  # [neuron d_model]

    # Get activations of the layer just after the activation function, i.e. this is f(x.T @ W_in)
    f_x_W_in = get_activation(model, data.toks, utils.get_act_name("post", layer))  # [batch seq neuron]

    # f_x_W_in are activations, so they have batch and seq dimensions - this is where we index by
    # sequence position if not None
    if seq is not None:
        f_x_W_in = f_x_W_in[:, seq, :]  # [batch neuron]

    # Calculate the output by neuron (i.e. so summing over the `neurons` dimension gives the output
    # of the MLP)
    out = einops.einsum(
        f_x_W_in,
        W_out,
        "... neuron, neuron d_model -> ... neuron d_model",
    )
    return out


def get_out_by_neuron_in_20_dir(
    model: HookedTransformer, data: BracketsDataset, layer: int
) -> Float[Tensor, "batch neurons"]:
    """
    [b, s, i]th element is the contribution of the vector written by the ith neuron to the residual stream in the
    unbalanced direction (for the b-th element in the batch, and the s-th sequence position).

    In other words we need to take the vector produced by the `get_out_by_neuron` function, and project it onto the
    unbalanced direction for head 2.0 (at seq pos = 1).
    """
    # Get neuron output at sequence position 1
    out_by_neuron_seqpos1 = get_out_by_neuron(model, data, layer, seq=1)

    # For each neuron, project the vector it writes to residual stream along the pre-2.0 unbalanced
    # direction
    return einops.einsum(
        out_by_neuron_seqpos1,
        get_pre_20_dir(model, data),
        "batch neuron d_model, d_model -> batch neuron",
    )
```
</details>

### 연습 문제 - 더 적은 메모리를 사용하여 동일한 함수 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You shouldn't spend more than 10-15 minutes on this exercise.
> Understanding the solution is more important than doing this exercise, so you should look at the solution rather than doing the exercise if you feel like it.
> ```

*이 연습 문제는 이전 문제만큼 중요하지 않으며, 흥미가 없다면 건너뛰어도 좋습니다. (하지만 여기서 어떤 일이 일어나는지 이해하기 위해 솔루션을 확인하는 것을 권장합니다.)*

만약 우리가 MLP에서 원하는 유일한 것이 불균형한 방향(unbalanced direction)에 대한 기여분뿐이라면, 사실 `out_by_neuron_in_20_dir` 객체를 저장하지 않고도 이를 수행할 수 있습니다. 이 방법을 찾아 아래에 구현해 보시기 바랍니다.

이러한 아이디어들은 toy 모델을 다룰 때는 필수적이지 않지만, 더 큰 모델을 다룰 때는 더욱 중요해지며, 메모리 제약 사항에 유의해야 합니다.

In [ ]:
def get_out_by_neuron_in_20_dir_less_memory(
    model: HookedTransformer, data: BracketsDataset, layer: int
) -> Float[Tensor, "batch neurons"]:
    """
    Has the same output as `get_out_by_neuron_in_20_dir`, but uses less memory (because it never
    stores the output vector of each neuron individually).
    """
    raise NotImplementedError()


tests.test_get_out_by_neuron_in_20_dir_less_memory(get_out_by_neuron_in_20_dir_less_memory, model, data_mini)

<details>
<summary>힌트</summary>

핵심은 연산 순서를 바꾸는 것입니다.

먼저, 각 output 방향을 pre-2.0 unbalanced 방향으로 project 하여 각 성분을 구합니다 (즉, 길이가 `d_mlp`인 벡터이며, 여기서 `i`번째 요소는 unbalanced 방향으로의 벡터 $W^{out}_{[i,:]}$의 성분입니다). 그 다음, 이 기여도들에 activation $f(\vec x^T W^{in}_{[:,i]} + b^{in}_i)$을 곱해 스케일을 조정합니다.**bold text**
</details>


<details><summary>솔루션</summary>

```python
def get_out_by_neuron_in_20_dir_less_memory(
    model: HookedTransformer, data: BracketsDataset, layer: int
) -> Float[Tensor, "batch neurons"]:
    """
    Has the same output as `get_out_by_neuron_in_20_dir`, but uses less memory (because it never
    stores the output vector of each neuron individually).
    """
    W_out = model.W_out[layer]  # [neurons d_model]

    f_x_W_in = get_activation(model, data.toks, utils.get_act_name("post", layer))[:, 1, :]  # [batch neurons]

    pre_20_dir = get_pre_20_dir(model, data)  # [d_model]

    # Multiply along the d_model dimension
    W_out_in_20_dir = W_out @ pre_20_dir  # [neurons]
    # Multiply elementwise, over neurons (we're broadcasting along the batch dim)
    out_by_neuron_in_20_dir = f_x_W_in * W_out_in_20_dir  # [batch neurons]

    return out_by_neuron_in_20_dir
```
</details>

### 뉴런 해석하기

이제 `2.0`에 특히 중요한 몇 개의 개별 뉴런을 식별해 보십시오.

예를 들어, 균형 잡힌 시퀀스와 불균형한 시퀀스(특히 열린 괄호로 시작하는 불균형한 시퀀스)에서 우리가 선택한 방향으로 쓰는 양의 차이가 가장 큰 뉴런이 무엇인지 확인하여 이를 수행할 수 있습니다.

`plot_neurons` 함수를 사용하여 개별 뉴런이 서로 다른 열린 괄호 비율에서 어떤 역할을 하는지 파악해 보십시오.

한 가지 참고 사항: 이제 우리는 네트워크의 내부 깊숙한 곳을 다루고 있으므로, 단일 방향이 이 전체-상승(overall-elevation) circuit에서 일어나는 의미 있는 일들의 대부분을 포착한다는 우리의 가정은 매우 의심스럽습니다. 이는 `2.0` 방향을 사용하여 `mlp0`의 출력을 분석할 때 특히 그러한데, 이 mlp가 영향을 미치는 주요 방법 중 하나는 우리가 방향을 설정하여 포착하려 한 경로가 아닌 더 간접적인 경로(예: `mlp0 -> mlp1 -> 2.0`)를 통해서이기 때문입니다. 따라서 서로 다른 layer나 뉴런이 무엇을 하고 있는지에 대해 얻는 직관이 불완전할 가능성이 높다는 점을 인지하는 것이 좋습니다.

*참고 - 플롯이 VSCode가 아닌 브라우저에서 열리도록 하는 기본 인자 `renderer="browser"`를 제공했습니다. 이는 특히 notebook에서 렉이 적어 더 잘 작동하는 경우가 많지만, 원하신다면 이를 제거하셔도 됩니다.*

In [ ]:
for layer in range(2):
    # Get neuron significances for head 2.0, sequence position #1 output
    neurons_in_unbalanced_dir = get_out_by_neuron_in_20_dir_less_memory(model, data, layer)[
        utils.to_numpy(data.starts_open), :
    ]

    # Plot neurons' activations
    plotly_utils.plot_neurons(neurons_in_unbalanced_dir, model, data, failure_types_dict, layer)

<details>
<summary>몇 가지 관찰 결과입니다:</summary>

layer 1의 중요한 neuron들은 크게 세 가지 범주로 나눌 수 있습니다:

- 일부 neuron들은 open-proportion이 1/2보다 클 때를 감지합니다. 몇 가지 예로, layer 1의 neuron **`1.53`**, **`1.39`**, **`1.8`**를 확인해 보십시오. **`0.33`**이나 **`0.43`**와 같이 layer 0에도 일부 존재합니다. 전반적으로 이러한 neuron들은 Layer 1에서 더 흔하게 나타나는 것으로 보입니다.

- 일부 neuron들은 open-proportion이 1/2보다 작을 때를 감지합니다. 예를 들어, neuron **`0.21`** 및 **`0.7`**가 이에 해당합니다. 이들은 layer 1에서 훨씬 더 드물게 나타나지만, **`1.50`** 및 **`1.6`**와 같은 사례를 볼 수 있습니다.

- 네트워크는 단순히 이 두 유형의 neuron만을 사용하고, 이들을 서로 더함으로써 open-proportion이 정확히 1/2와 같은지를 측정하도록 조합할 수 있습니다. 하지만 layer 1에서는 이러한 조합된 속성을 출력하는 많은 neuron들이 존재함을 알 수 있습니다. 몇 가지 예로 **`1.10`**와 **`1.3`**을 확인해 보십시오.
    - ReLU는 단조(monotonic) 함수이며, open-paren proportion에 대해 비단조 함수 형태의 출력이 필요하기 때문에, layer 0의 단일 neuron이 스스로 이를 수행하는 것은 훨씬 더 어렵습니다. 하지만 **`mlp0`** 이전의 layernorm을 활용하여 이를 근사하는 것은 가능하며, **`0.19`**와 **`0.34`**이 이에 대한 좋은 예시입니다.

참고로, 반대 방향으로 작동하는 것으로 보이는 일부 neuron들이 있습니다 (예: `0.0`). 이러한 neuron들의 정확한 기능이 무엇인지는 불분명합니다 (특히 우리가 모델 circuit의 특정 부분만 분석하고 있으므로, 특정 neuron이 하는 일에 대한 우리의 직관이 불완전할 수 있습니다). 하지만 이 plot에서 명확하고 모호하지 않게 알 수 있는 점은, 우리의 neuron들이 괄호의 open proportion을 감지하고 있으며, 그 비율이 1/2보다 엄격히 크거나 작은지에 따라 다르게 반응한다는 것입니다. 그리고 이들 중 상당수가 head `2.0`에서 복사됨으로써 주요한 영향을 미친다는 것을 알 수 있습니다.

---

아래는 neuron **`0.21`**와 **`1.53`**의 plot입니다. 위에서 설명한 패턴들을 관찰할 수 있습니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/n21.png" width="550">
<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/n53.png" width="550">
</details>

## open-proportion이 어떻게 계산되는지 이해하기 - Head 0.0

지금까지 우리는 logit부터 시작하여 네트워크 내부로 거슬러 올라가며 분석해 왔습니다. 이제는 전략을 조금 바꾸어, 입력 embedding부터 순방향으로 분석해 보겠습니다. 특히, 네트워크가 처음에 시퀀스의 open-proportion을 어떻게 계산하는지 이해하고자 합니다!

그 핵심은 head 0.0이 될 것입니다. 먼저 이 head의 attention pattern을 살펴보는 것부터 시작하겠습니다.

### 0.0 Attention Pattern

우리는 head들의 attention pattern을 다양하게 실험해 보고 싶습니다. 예를 들어, "query가 항상 왼쪽 괄호일 때 attention pattern은 어떤 모습일까?"와 같은 질문을 던져보고 싶습니다. 이를 위해 괄호 문자열을 입력받아 `q` 및 `k` 벡터(즉, attention score를 얻기 위해 내적하는 값들)를 반환하는 함수를 작성하겠습니다.

### 연습 문제 - hook을 사용하여 query와 key 추출하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You shouldn't spend more than ~15 minutes on this exercise.
> Again, this exercise just involves using your `get_activations` function.
> ```

In [ ]:
def get_q_and_k_for_given_input(
    model: HookedTransformer,
    tokenizer: SimpleTokenizer,
    parens: str,
    layer: int,
) -> tuple[Float[Tensor, "seq n_heads d_model"], Float[Tensor, "seq n_heads d_model"]]:
    """
    Returns the queries and keys for the given parens string, for all attn heads in the given layer.
    """
    raise NotImplementedError()


tests.test_get_q_and_k_for_given_input(get_q_and_k_for_given_input, model, tokenizer)

<details><summary>솔루션</summary>

```python
def get_q_and_k_for_given_input(
    model: HookedTransformer,
    tokenizer: SimpleTokenizer,
    parens: str,
    layer: int,
) -> tuple[Float[Tensor, "seq n_heads d_model"], Float[Tensor, "seq n_heads d_model"]]:
    """
    Returns the queries and keys for the given parens string, for all attn heads in the given layer.
    """
    q_name = utils.get_act_name("q", layer)
    k_name = utils.get_act_name("k", layer)

    activations = get_activations(model, tokenizer.tokenize(parens), [q_name, k_name])

    return activations[q_name][0], activations[k_name][0]
```
</details>

### Activation Patching

이제 매우 유용한 도구인 **activation patching**을 소개하겠습니다. 이는 [David Bau and Kevin Meng's excellent ROME paper](https://rome.baulab.info/)에서 처음 소개되었으며, 그곳에서는 causal tracing이라고 불렸습니다.

activation patching의 설정은 두 가지 서로 다른 입력에 대해 모델을 두 번 실행하는 것입니다. 하나는 clean run이고 다른 하나는 corrupted run입니다. clean run은 정답을 출력하고, corrupted run은 그렇지 않습니다. 핵심 아이디어는 모델에 corrupted 입력을 제공하되, 특정 activation에 **개입(intervene)**하여 clean run의 해당 activation을 **패치(patch)**하고(즉, corrupted activation을 clean activation으로 교체), 실행을 계속하는 것입니다.

activation patching의 일반적인 사용 사례 중 하나는 clean run과 patched run에서의 모델 성능을 비교하는 것입니다. 패칭 후 성능이 저하된다면, 이는 패치를 수행한 위치가 모델의 계산에 중요하다는 강력한 신호입니다. 위치를 국소화(localise)하는 능력은 mechanistic interpretability의 핵심 단계입니다. 만약 계산이 모델 전체에 흩어져 있고 분산되어 있다면, 어떤 일이 일어나고 있는지에 대해 명확한 mechanistic 이야기를 구성하기가 훨씬 더 어려울 가능성이 큽니다. 하지만 모델의 어느 부분이 중요한지 정확하게 식별할 수 있다면, 해당 부분을 확대하여 그것들이 무엇을 나타내고 서로 어떻게 연결되는지 결정할 수 있으며, 궁극적으로 그것들이 나타내는 기저의 circuit을 역공학(reverse engineer)할 수 있습니다.

하지만 여기서 우리의 path patching은 훨씬 더 단순한 목적을 위해 사용됩니다. 우리는 head `0.0`의 **query 벡터**에 모든 왼쪽 괄호 시퀀스의 값들을 패치하고, **key 벡터**에는 모든 왼쪽 및 오른쪽 괄호의 평균값들을 패치할 것입니다. 이를 통해 왼쪽 괄호가 시퀀스의 나머지 부분에 기울이는 평균적인 attention 패턴을 파악할 수 있습니다.

layer 0의 두 head 모두에 대해 이를 수행하는 함수를 작성하겠습니다. 두 head를 비교하는 것이 유익하기 때문입니다.

In [ ]:
layer = 0
all_left_parens = "".join(["(" * 40])
all_right_parens = "".join([")" * 40])

model.reset_hooks()
q0_all_left, k0_all_left = get_q_and_k_for_given_input(model, tokenizer, all_left_parens, layer)
q0_all_right, k0_all_right = get_q_and_k_for_given_input(model, tokenizer, all_right_parens, layer)
k0_avg = (k0_all_left + k0_all_right) / 2

# Define hook function to patch in q or k vectors
def hook_fn_patch_qk(
    value: Float[Tensor, "batch seq head d_head"],
    hook: HookPoint,
    new_value: Float[Tensor, "... seq d_head"],
    head_idx: int | None = None,
) -> None:
    if head_idx is not None:
        value[..., head_idx, :] = new_value[..., head_idx, :]
    else:
        value[...] = new_value[...]


# Define hook function to display attention patterns (using plotly)
def hook_fn_display_attn_patterns(
    pattern: Float[Tensor, "batch heads seqQ seqK"], hook: HookPoint, head_idx: int = 0
) -> None:
    avg_head_attn_pattern = pattern.mean(0)
    labels = ["[start]", *[f"{i + 1}" for i in range(40)], "[end]"]
    display(
        cv.attention.attention_heads(
            tokens=labels,
            attention=avg_head_attn_pattern,
            attention_head_names=["0.0", "0.1"],
            max_value=avg_head_attn_pattern.max(),
            mask_upper_tri=False,  # use for bidirectional models
        )
    )


# Run our model on left parens, but patch in the average key values for left vs right parens
# This is to give us a rough idea how the model behaves on average when the query is a left paren
model.run_with_hooks(
    tokenizer.tokenize(all_left_parens).to(device),
    return_type=None,
    fwd_hooks=[
        (utils.get_act_name("k", layer), partial(hook_fn_patch_qk, new_value=k0_avg)),
        (utils.get_act_name("pattern", layer), hook_fn_display_attn_patterns),
    ],
)

<details>
<summary>질문 - 이 플롯에서 head <code>0.0</code>의 주목할 만한 특징은 무엇입니까?</summary>

가장 주목할 만한 특징은 대각선 패턴입니다. 대부분의 query token은 자신보다 앞에 오는 모든 token에는 거의 zero attention을 기울이지만, 뒤에 오는 token들에는 훨씬 더 큰 attention을 기울입니다. 대부분의 query token 위치에서, 자신 이후의 token들에 기울이는 이 attention은 대략적으로 균일합니다. 하지만 (특히 나중의 query 위치에서) 자신 이후의 token들에 기울이는 attention이 균일하지 않은 몇몇 패치들이 존재합니다. 이러한 패치들이 adversarial examples를 생성하는 데 중요하다는 것을 알게 될 것입니다.

query가 오른쪽 괄호일 때도 대략적으로 동일한 패턴을 관찰할 수 있습니다 (위의 마지막 코드 부분을 실행하되, `all_left_parens` 대신 `all_right_parens`를 사용해 보십시오). 다만 패턴이 덜 뚜렷합니다.
</details>

우리는 query 위치 1에서의 attention pattern에 가장 관심이 많습니다. 왜냐하면 이곳이 정보를 이동시켜 결국 attention head `2.0`로 전달하고, 다시 위치 0으로 이동시켜 예측에 사용하는 위치이기 때문입니다.

(참고 - 우리는 첫 번째 괄호가 여는 괄호인 시나리오에 집중하기로 했습니다. 모델이 오른쪽 괄호로 시작하는 bracket string을 처리하는 방식은 약간 다르기 때문입니다. 이러한 문자열은 분명히 불균형하므로 복잡한 메커니즘이 필요하지 않습니다.)

위치 1에 있는 여는 괄호 query가 다른 모든 위치에 기울이는 attention probability를 막대 그래프로 그려보겠습니다. 여기서는 인위적인 시퀀스에서 key와 query를 모두 patching 하는 대신, 전체 데이터셋에 대해 모델을 실행하고 query에 대해서만 인위적인 값(모든 여는 괄호)을 patching 합니다. 위치 1의 query 벡터가 여는 괄호일 때 어떻게 동작하는지에 대한 일반적인 감각을 찾는 것이므로, 두 방법 모두 합리적입니다.

In [ ]:
def hook_fn_display_attn_patterns_for_single_query(
    pattern: Float[Tensor, "batch heads seqQ seqK"],
    hook: HookPoint,
    head_idx: int = 0,
    query_idx: int = 1,
):
    bar(
        utils.to_numpy(pattern[:, head_idx, query_idx].mean(0)),
        title="Average attn probabilities on data at posn 1, with query token = '('",
        labels={"index": "Sequence position of key", "value": "Average attn over dataset"},
        height=500,
        width=800,
        yaxis_range=[0, 0.1],
        template="simple_white",
    )


data_len_40 = BracketsDataset.with_length(data_tuples, 40).to(device)

model.reset_hooks()
model.run_with_hooks(
    data_len_40.toks[data_len_40.isbal],
    return_type=None,
    fwd_hooks=[
        (utils.get_act_name("q", 0), partial(hook_fn_patch_qk, new_value=q0_all_left)),
        (utils.get_act_name("pattern", 0), hook_fn_display_attn_patterns_for_single_query),
    ],
)

<details>
<summary>질문 - 이 attention pattern의 해석은 무엇입니까?</summary>

이는 attention pattern이 모든 token에 대해 거의 정확하게 균등함을 보여줍니다. 이는 sequence position 1에 기록되는 벡터가 각 source position에 있는 벡터들의 합에 행렬 $W_{OV}^{0.0}$을 통해 변환된 값의 대략적인 스칼라 배가 됨을 의미합니다.
</details>

### 가설 제안하기

모든 조각을 하나로 연결하기 전에, 지금까지 우리 모델에 대해 알게 된 사실들을 (관찰한 순서대로) 나열해 보겠습니다:

> * Attention head `2.0`는 괄호들의 net elevation이 0이 아닐 때(즉, 왼쪽 괄호와 오른쪽 괄호의 개수가 다를 때) 이를 unbalanced로 분류하는 데 주로 책임이 있는 것으로 보입니다.
    * Attention head `2.0`는 sequence position $i=1$에 강하게 attend합니다. 다시 말해, 이는 거의 단순히 position 1의 residual stream 벡터를 position 0으로 이동시키는 것(그리고 matrix $W_{OV}$를 적용하는 것)과 같습니다.
    * 따라서 모델의 더 앞선 컴포넌트들이 sequence position 1에 정보를 기록하여, (head `2.0`를 통하는 경로를 통해) 모델이 올바른 분류를 하도록 영향을 주고 있음이 분명합니다.
* `MLP0`와 `MLP1`에는 열린 괄호 비율의 nonlinear 함수를 계산하는 것으로 보이는 여러 neuron이 있습니다. 일부는 비율이 $1/2$보다 엄격히 클 때 강하게 activation되고, 다른 일부는 $1/2$보다 엄격히 작을 때 activation됩니다.
* Attention head `0.0`의 query token이 열린 괄호인 경우, $i$ **이후**의 모든 key position에 거의 동일한 크기로 attend합니다.
    * 특히, 이는 모든 sequence position에 대략 균일하게 attend하는 sequence position $i=1$에 대해서도 성립합니다.

이 모든 내용을 바탕으로, 이 세 가지 관찰 결과를 하나로 묶어 elevation circuit이 어떻게 작동하는지에 대한 가설을 세울 수 있을까요?

<details>
<summary>가설</summary>

가설은 다음과 같은 식이 될 수 있습니다:

1. **head `0.0`의 attention 계산에서, position-1 query token은 괄호들에 대해 일종의 aggregation을 수행합니다. 이는 왼쪽 괄호와 오른쪽 괄호 개수의 차이, 즉 net elevation을 나타내는 정보를 residual stream에 기록합니다.**
> 단일 레이어 attention head는 기본적으로 `keep ... in -> mind`과 같은 형태의 skip-trigram만 수행할 수 있다는 점을 기억하십시오. 이들은 세 가지 요소 간의 상호작용을 유연하게 포착할 수 없으며, 다시 말해 "왼쪽과 오른쪽 괄호의 개수가 같은지 여부"와 같은 함수를 계산할 수 없습니다. (이를 더 명확히 하기 위해, 모델이 단일 레이어일 때 입력 `()`, `((`, `))`에서 동작이 어떻게 달라질지 생각해보십시오). 따라서 왼쪽과 오른쪽 괄호에 대한 aggregation이 우리가 할 수 있는 거의 전부입니다.

2. **이제 sequence position 1에 elevation에 대한 정보가 포함되었으므로, MLP가 이 정보를 읽고 일부 neuron이 nonlinear 연산을 수행하여 왼쪽과 오른쪽 괄호의 개수가 같은지에 대한 "boolean" 정보를 포함하는 벡터를 생성합니다.**
> MLP는 선형 함수(예: 왼쪽과 오른쪽 괄호 개수의 차이)를 받아 이를 boolean 정보로 변환하는 데 매우 뛰어나다는 점을 상기하십시오. 위에서 본 plot들에서도 대부분의 MLP neuron 동작이 왼쪽 괄호 비율 50% 임계값을 기준으로 뚜렷하게 달랐으므로, 이와 유사한 일이 일어나고 있음을 확인했습니다.

3. **마지막으로, residual stream의 첫 번째 sequence position에 net elevation이 0인지에 대한 boolean 정보가 저장되었으므로, 이 정보는 head `2.0`에 의해 읽히며, 이 head의 출력은 시퀀스를 balanced 또는 unbalanced로 분류하는 데 사용됩니다.**
> 이는 head `2.0`가 첫 번째 sequence position에 강하게 attend하고 있으며, elevation 테스트를 구현하고 있는 것으로 보인다는 사실에 근거합니다.
</details>

이 시점에서 우리는 위의 모든 관찰 결과들을 거의 경험적으로 검증했습니다. 아직 제대로 증명하지 못한 한 가지는 **(1)**이 위에서 설명한 대로 작동하고 있다는 점입니다. 우리는 head `0.0`가 왼쪽과 오른쪽 괄호 개수 사이의 일종의 차이를 계산하고, 이 정보를 residual stream에 기록하고 있는지 확인하고 싶습니다. 다음 섹션에서 이 가설을 테스트할 방법을 찾아보겠습니다.

### 0.0 OV circuit

**우리는 `0.0` head가 residual stream에 무엇을 쓰고 있는지 이해하고자 합니다. 특히, net elevation에 대한 정보를 쓰고 있다는 증거를 찾고 있습니다.**

우리는 이미 query position 1이 모든 key position에 거의 균등하게 attention을 주고 있다는 것을 확인했습니다. 이는 (시작 및 종료 token을 제외하면) position 1에 쓰이는 벡터가 대략 다음과 같음을 의미합니다:

$$
\begin{aligned}
h(x) &\approx \frac{1}{n} \sum_{i=1}^n \left(\left(L {\color{orange}{x}}\right)^T W_{OV}^{0.0}\right)_i \\
&= \frac{1}{n} \sum_{i=1}^n {\color{orange}{x_i}}^T L^T W_{OV}^{0.0} \\
\end{aligned}
$$

여기서 $L$는 첫 번째 attention layer 이전의 layernorm에 대한 선형 근사치이며, $x$은 각 sequence position $i$에 대한 벡터 ${\color{orange}{x_i}}$로 구성된 `(seq_len, d_model)` 크기의 residual stream입니다.

우리는 ${\color{orange}{x_j}} = {\color{orange}{pos_j}} + {\color{orange}{tok_j}}$로 쓸 수 있으며, 여기서 ${\color{orange}{pos_j}}$과 ${\color{orange}{tok_j}}$는 각각 positional embedding과 token embedding을 나타냅니다. 따라서 다음과 같은 식을 얻을 수 있습니다:

$$
\begin{aligned}
h(x) &\approx \frac{1}{n} \left( \sum_{i=1}^n {\color{orange}{pos_i}}^T L^T W_{OV}^{0.0} + \sum_{i=1}^n {\color{orange}{tok_i}}^T L^T W_{OV}^{0.0}\right) \\
&= \frac{1}{n} \left( \sum_{i=1}^n {\color{orange}{pos_i}}^T L^T W_{OV}^{0.0} + n_L {\color{orange}{\vec v_L}} + n_R {\color{orange}{\vec v_R}}\right)
\end{aligned}
$$

여기서 $n_L$과 $n_R$는 각각 왼쪽 괄호와 오른쪽 괄호의 개수이며, ${\color{orange}{\vec v_L}}, {\color{orange}{\vec v_R}}$은 layernorm과 OV circuit의 이미지 하에서 각각 왼쪽 및 오른쪽 괄호 token embedding의 이미지입니다:

$$
\begin{aligned}
{\color{orange}{\vec v_L}} &= {\color{orange}{LeftParen}}^T L^T W_{OV}^{0.0} \\
{\color{orange}{\vec v_R}} &= {\color{orange}{RightParen}}^T L^T W_{OV}^{0.0}
\end{aligned}
$$

여기서 ${\color{orange}{LeftParen}}$와 ${\color{orange}{RightParen}}$은 각각 왼쪽 및 오른쪽 괄호의 token embedding입니다.

마지막으로, 위의 식을 통해 우리의 가설을 검증하기 위한 테스트를 공식화할 수 있습니다:

> 만약 head `0.0`가 일종의 aggregation을 수행하고 있다면, **${\color{orange}\vec v_L}$과 ${\color{orange}\vec v_R}$가 서로 반대 방향을 가리키는 벡터임을 확인할 수 있어야 합니다.** 다시 말해, head `0.0`는 벡터 $v$의 어떤 스칼라 배수를 residual stream에 쓰고, 우리는 이 벡터의 방향으로 projection함으로써 정보 $n_L - n_R$를 추출할 수 있습니다. 그러면 MLP는 이 정보를 가져와 비선형 방식으로 처리하고, 시퀀스가 balanced 되었는지에 대한 정보를 residual stream에 씁니다.

### 연습 문제 - 가설 검증하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You shouldn't spend more than 10-15 minutes on this exercise.
> If you understand what the vectors represent, these exercises should be pretty straightforward.
> ```

여기서 여러분은 두 벡터의 cosine similarity가 -1에 가깝다는 것을 보여줌으로써, 이 head가 이후에 나오는 여는 괄호와 닫는 괄호를 "집계(tallying)"하고 있음을 증명해야 합니다.

특정 문자에 해당하는 token embedding 벡터(즉, 위에서 ${\color{orange}LeftParen}$ 및 ${\color{orange}RightParen}$라고 불렀던 벡터들)를 반환하는 함수 `embedding`를 작성하면, 이러한 벡터들을 계산할 때 도움이 될 것입니다.

In [ ]:
def embedding(model: HookedTransformer, tokenizer: SimpleTokenizer, char: str) -> Float[Tensor, "d_model"]:
    assert char in ("(", ")")
    idx = tokenizer.t_to_i[char]
    return model.W_E[idx]


# YOUR CODE HERE - define v_L and v_R, as described above.

print(f"Cosine similarity: {t.cosine_similarity(v_L, v_R, dim=0).item():.4f}")

<details>
<summary>두 벡터에 관한 추가적인 기술적 세부 사항 (선택 사항)</summary>

참고 - 이 아이디어가 작동하기 위해 $\color{orange}{\vec v_L}$과 $\color{orange}{\vec v_R}$가 반드시 동일한 크기를 가질 필요는 없습니다. 왜냐하면 어떤 $\alpha > 0$에 대해 ${\color{orange} \vec v_L} \approx - \alpha {\color{orange} \vec v_R}$를 가지고 있다면, $\color{orange}{\vec v_L}$ 방향으로 projection 했을 때 $\|{\color{orange} \vec v_L}\| (n_L - \alpha n_R) / n$을 얻게 되기 때문입니다. 이는 시퀀스 길이에 관계없이 왼쪽과 오른쪽 괄호의 수가 일치할 때 항상 $\|{\color{orange} \vec v_L}\| (1 - \alpha) / 2$와 같습니다. 이 값이 0이 아니라는 점은 중요하지 않습니다. MLP의 neuron들은 bias 항을 추가함으로써 이 방향의 벡터 성분이 이 값보다 큰지 작은지를 감지하도록 여전히 학습할 수 있습니다. 중요한 점은 (1) 두 벡터가 평행하며 서로 반대 방향을 향하고 있다는 것과, (2) *균형 잡힌 시퀀스*에 대해 이 방향으로의 projection 결과가 항상 동일하다는 것입니다.

</details>


<details><summary>솔루션</summary>

```python
W_OV = model.W_V[0, 0] @ model.W_O[0, 0]

layer0_ln_fit = get_ln_fit(model, data, layernorm=model.blocks[0].ln1, seq_pos=None)[0]
layer0_ln_coefs = t.from_numpy(layer0_ln_fit.coef_).to(device)

v_L = embedding(model, tokenizer, "(") @ layer0_ln_coefs.T @ W_OV
v_R = embedding(model, tokenizer, ")") @ layer0_ln_coefs.T @ W_OV

print(f"Cosine similarity: {t.cosine_similarity(v_L, v_R, dim=0).item():.4f}")
```
</details>

### 연습 문제 - 입력 방향의 코사인 유사도 (선택 사항)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You shouldn't spend more than 10-15 minutes on this exercise.
> ```

이 가설에 대한 증거를 얻는 또 다른 방법이 있습니다. MLP neuron에 대한 논의에서 $W^{in}_{[:,i]}$ ($W^{in}$가 MLP의 첫 번째 linear layer일 때, 행렬 $W^{in}$의 $i$번째 열)이 neuron의 "in-direction"을 나타내는 벡터라는 점을 상기해 보십시오. 만약 이 neuron들이 우리가 생각하는 방식으로 open/closed 비율을 측정하고 있다면, 벡터 $v_R$, $v_L$이 이 벡터들과 높은 내적(dot product) 값을 가질 것으로 기대할 수 있습니다.

아래의 두 함수를 완성하여 이를 조사해 보십시오. `cos_sim_with_MLP_weights`은 특정 layer의 $W^{in}$의 열들과 주어진 벡터 사이의 코사인 유사도 벡터를 반환하며, `avg_squared_cos_sim`는 벡터 $v$과 $v$과 동일한 크기를 가진 무작위로 선택된 벡터 사이의 평균 **제곱 코사인 유사도(squared cosine similarity)**를 반환합니다 (이 벡터는 iid 정규 분포에서 샘플링한 후 정규화하는 등 합리적인 방식으로 선택할 수 있습니다). $v_R$와 `MLP0` 및 `MLP1`에 있는 neuron들의 in-direction 사이의 neuron당 평균 제곱 코사인 유사도가 우연히 발생할 것으로 예상되는 값보다 훨씬 높다는 것을 발견하게 될 것입니다.

In [ ]:
def cos_sim_with_MLP_weights(
    model: HookedTransformer, v: Float[Tensor, "d_model"], layer: int
) -> Float[Tensor, "d_mlp"]:
    """
    Returns a vector of length d_mlp, where the ith element is the cosine similarity between v and
    the ith in-direction of the MLP in layer `layer`.

    Recall that the in-direction of the MLPs are the columns of the W_in matrix.
    """
    raise NotImplementedError()


def avg_squared_cos_sim(v: Float[Tensor, "d_model"], n_samples: int = 1000) -> float:
    """
    Returns the average (over n_samples) cosine similarity between v and another randomly chosen
    vector of length `d_model`.

    We can create random vectors from the standard N(0, I) distribution.
    """
    raise NotImplementedError()


print("Avg squared cosine similarity of v_R with ...\n")

cos_sim_mlp0 = cos_sim_with_MLP_weights(model, v_R, 0)
print(f"...MLP input directions in layer 0: {cos_sim_mlp0.pow(2).mean():.4f}")

cos_sim_mlp1 = cos_sim_with_MLP_weights(model, v_R, 1)
print(f"...MLP input directions in layer 1: {cos_sim_mlp1.pow(2).mean():.4f}")

cos_sim_rand = avg_squared_cos_sim(v_R)
print(f"...random vectors of len = d_model: {cos_sim_rand:.4f}")

<details><summary>솔루션</summary>

```python
def cos_sim_with_MLP_weights(
    model: HookedTransformer, v: Float[Tensor, "d_model"], layer: int
) -> Float[Tensor, "d_mlp"]:
    """
    Returns a vector of length d_mlp, where the ith element is the cosine similarity between v and
    the ith in-direction of the MLP in layer `layer`.

    Recall that the in-direction of the MLPs are the columns of the W_in matrix.
    """
    v_unit = v / v.norm()
    W_in_unit = model.W_in[layer] / model.W_in[layer].norm(dim=0)

    return einops.einsum(v_unit, W_in_unit, "d_model, d_model d_mlp -> d_mlp")


def avg_squared_cos_sim(v: Float[Tensor, "d_model"], n_samples: int = 1000) -> float:
    """
    Returns the average (over n_samples) cosine similarity between v and another randomly chosen
    vector of length `d_model`.

    We can create random vectors from the standard N(0, I) distribution.
    """
    v2 = t.randn(n_samples, v.shape[0]).to(device)
    v2 /= v2.norm(dim=1, keepdim=True)

    v1 = v / v.norm()

    return (v1 * v2).pow(2).sum(1).mean().item()
```
</details>

_추가_ 보너스 연습 문제로, 뉴런당 squared cosine similarity를 이전에 작성한 뉴런 기여도 플롯(슬라이더가 있는 플롯)과 비교해 볼 수 있습니다. $v_R$와 특히 높은 cosine similarity를 가진 뉴런들이, 열린 괄호의 비율이 0.5가 아닐 때마다 head `2.0`의 unbalanced direction으로 크게 write하는 뉴런들과 일치합니까? (이는 net elevation circuit에서 사용되는 전체 열린 괄호 비율에 대한 정보의 주요 소스가 head `0.0`에 의해 residual stream에 write되는 $v_R$와 $v_L$의 배수들에 의해 제공된다는 추가적인 증거가 될 것입니다). 이전 플롯으로 돌아가서 확인해 보시기 바랍니다.

## 요약

좋습니다! 잠시 멈춰서 이 circuit에 대해 배운 내용을 정리해 보겠습니다.

> Head 0.0은 각 token 뒤에 오는 suffix에 균일하게 attention을 기울여, 발견하는 열린 괄호와 닫힌 괄호의 개수를 합산하고 그 값을 residual stream에 씁니다. 이는 전체 elevation을 나타내는 벡터를 residual stream 1에 쓴다는 것을 의미합니다. 그 후 residual stream 1의 MLP들은 이 합산 값에 대해 비선형적으로 작동하여, 전체 elevation이 0인 경우와 0이 아닌 경우를 구분하는 벡터를 residual stream에 씁니다. Head 2.0은 이 신호를 residual stream 0으로 복사하며, 여기서 신호는 classifier를 거쳐 unbalanced로 분류되게 합니다. 이 동작에 대한 우리의 1차적인 이해가 완료되었습니다.

이 circuit의 일러스트레이션이 아래에 제공됩니다. 많은 구성 요소가 포함되어 있어 꽤 복잡하므로, 모든 내용을 다 이해하지 못하더라도 걱정하지 마십시오!

범례: 굵은 검은색 선과 주황색 점선은 elevation circuit을 구성하는 transformer 내부의 경로를 보여줍니다. 주황색 점선은 skip connection을 나타냅니다. 중요한 head와 MLP layer들은 각각 굵은 색상으로 표시되어 있습니다. 우리 circuit의 세 가지 중요한 부분(head `0.0`, MLP layer들, 그리고 head `2.0`)에는 각각 어떤 역할을 하는지와 우리가 발견한 근거를 설명하는 주석이 달려 있습니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/bracket-transformer-attribution-2.png" width="1150">

# ☆ 보너스

> ##### 학습 목표
>
> * 모델의 작동 방식에 대한 이해를 바탕으로 adversarial examples를 생성합니다.
> * 모델의 특정 이상 특징(anomalous features)에 대해 더 깊이 탐구합니다.

## bracket transformer 조사하기

여기에는 이전 내용을 바탕으로 한 몇 가지 보너스 연습 문제가 있습니다 (예: 모델의 서로 다른 부분을 조사하거나, 모델의 작동 방식에 대한 이해를 바탕으로 adversarial examples를 생성하는 것).

*이 마지막 섹션은 가이드가 적은 편이지만, 제안된 연습 문제들은 이전 섹션과 비슷한 성격입니다.*

> ##### 학습 목표
>
> * 모델의 작동 방식에 대한 이해를 사용하여 adversarial examples를 생성합니다.
> * 모델의 특정 이상 특징(anomalous features)에 대해 더 깊이 탐구합니다.

저희가 추천하는 주요 보너스 연습 문제는 **adversarial attacks**입니다. classification circuit의 나머지 절반이 어떻게 작동하는지 파악하기 위해 **detecting anywhere-negative failures** 보너스 연습 문제의 첫 번째 섹션을 읽어야 하지만, 이 내용을 이해하고 나면 바로 adversarial attacks 섹션으로 넘어가셔도 됩니다.

### anywhere-negative failure 감지하기

attention pattern 그리드를 살펴보았을 때, 첫 번째 query token이 그 뒤에 오는 모든 token에 거의 균일한 attention을 기울일 뿐만 아니라, 다른 대부분의 token들도 (정도는 낮지만) 마찬가지라는 것을 확인했습니다. 이는 $i$ 위치(일반적인 $i\geq 1$ 에 대해)에 기록되는 벡터를 다음과 같이 쓸 수 있음을 의미합니다:

$$
\begin{aligned}
h(x)_i &\approx \frac{1}{n-i+1} \sum_{j=i}^n {\color{orange} x_j}^T L^T W_{OV}^{0.0} \\
&= \frac{1}{n} \left( \sum_{i=1}^n {\color{orange} pos_i}^T L^T W_{OV}^{0.0} + n_L^{(i)} {\color{orange}\vec v_L} + n_R^{(i)} {\color{orange}\vec v_R}\right)
\end{aligned}
$$

여기서 $n_L^{(i)}$ 와 $n_R^{(i)}$ 는 `brackets[i: n]` 로부터 형성된 substring에 포함된 왼쪽 및 오른쪽 괄호의 개수입니다 (즉, 이는 $i=1$ 일 때 $n_L$ 및 $n_R$ 의 정의와 일치합니다).

지금까지 확인한 내용(시퀀스 위치 1이 시퀀스 내 모든 괄호에 대한 tally 정보를 저장한다는 점)을 바탕으로, 각 시퀀스 위치가 유사한 tally를 저장하며, 이는 해당 위치 오른쪽에 있는 모든 괄호로 구성된 substring에 elevation failure가 있는지 판단하는 데 사용된다고 추측할 수 있습니다 (즉, ***오른쪽*** 괄호의 총 개수가 ***왼쪽*** 괄호의 총 개수보다 최소한 같거나 많은지 확인하는 것입니다. 우리 모델이 동일하게 유효한 우-좌(right-to-left) 솔루션을 학습했기 때문에 방향이 이렇다는 점을 기억하십시오).

destination token은 source에 얼마나 많은 attention을 기울일지만 결정하며, attention이 기울어졌을 때 source에서 destination으로 이동하는 벡터는 모든 destination token에 대해 동일하다는 점을 기억하십시오. 따라서 left-paren 벡터와 right-paren 벡터의 cosine similarity가 -1이라는 결과는 이후의 모든 시퀀스 위치에서도 동일하게 적용됩니다.

**Head 2.1은 anywhere-negative failure를 감지하는 head로 밝혀졌습니다** (즉, 임의의 시퀀스 `brackets[i: n]` 가 왼쪽 괄호보다 오른쪽 괄호를 엄격하게 더 많이 가지고 있는지 감지하며, 이 경우 불균형한 방향으로 residual stream에 기록합니다). 이러한 동작에 대한 증거를 찾을 수 있습니까?

이를 조사하는 한 가지 방법은 특정 지점에서 "음수로 전환되는(goes negative)" 괄호 문자열을 구성하고, destination 위치 0에서 head 2.0의 attention 확률을 살펴보는 것입니다. 괄호가 음수로 전환되는 source token들에 가장 강하게 attend하며, residual stream에 기록되는 해당 벡터가 불균형한 방향을 가리키고 있습니까?

또한 head 2.0에서 했던 것처럼 head 2.1의 입력들을 살펴볼 수 있습니다. 어떤 성분들이 가장 중요하며, 그 이유는 무엇이라고 추측하십니까?

<details>
<summary>정답</summary>

MLP가 head 2.1의 중요한 입력이라는 것을 발견하셨을 것입니다. 이는 타당한 결과입니다. 왜냐하면 앞서 MLP가 시퀀스 위치 1에서 tally 정보 $(n_L - \alpha n_R)$ 를 boolean 정보 $(n_L = n_R)$ 로 변환하는 것을 보았기 때문입니다. MLP는 모든 시퀀스 위치에서 동일하게 작동하므로, 각 시퀀스 위치 $i$ 에 boolean 정보 $(n_L^{(i)} > n_R^{(i)})$ 를 저장하고 있다고 추측하는 것이 합리적이며, 이것이 바로 anywhere-negative failure를 감지하는 데 필요한 정보입니다.
</details>

### Adversarial attacks

우리가 사용해 온 데이터셋에서 우리 모델은 약 10,000개 중 1개의 예제에 대해 오답을 냅니다. 모델에 대한 이해를 바탕으로, 수동으로 오분류된 입력을 찾을 수 있을까요? 지금 읽기를 멈추고 지금까지 배운 내용을 적용하여 오분류된 sequence를 직접 찾아보시는 것을 추천합니다. 만약 잘 되지 않는다면, 몇 가지 힌트를 확인해 보시기 바랍니다.

In [ ]:
adversarial_examples = ["()", "(())", "))"]


# YOUR CODE HERE - update the `adversarial_examples` list, to find adversarial examples!

m = max(len(ex) for ex in adversarial_examples)
toks = tokenizer.tokenize(adversarial_examples)
probs = model(toks)[:, 0].softmax(-1)[:, 1]
print("\n".join([f"{ex:{m}} -> {p:.4%} balanced confidence" for (ex, p) in zip(adversarial_examples, probs)]))

<details>
<summary>힌트 1</summary>

attention pattern의 오른쪽 하단 구석에 있는 저 이상한 패치 형태의 부분들은 무엇일까요? 이것을 이용할 수 있을까요?

더 구체적인 방향을 확인하려면 다음 힌트를 읽어주십시오.
</details>

<details>
<summary>힌트 2</summary>

우리는 각 왼쪽 괄호가 오른쪽에 있는 각 token에 거의 균등하게 attention을 기울인다는 점을 관찰했으며, 이를 통해 어느 지점에서든 elevation failure를 감지했습니다. 또한, 이러한 거의 균등한 패턴이 query 위치 27-31 주변에서 무너진다는 점을 알고 있습니다.

이를 염두에 두고, 모델이 balanced로 분류하게 만들 수 있는 "아슬아슬하게" 불균형한 괄호 문자열을 어떻게 구성할 수 있을까요?

추천하는 괄호 문자열 유형을 확인하려면 다음 힌트를 읽어주십시오.
</details>

<details>
<summary>힌트 3</summary>

우리는 어느 한 지점에서는 negative elevation을 가지지만, 그 외의 모든 곳에서는 balanced인 문자열을 구성하고자 합니다. 이는 `A)(B` 형태의 시퀀스를 사용하여 수행할 수 있으며, 여기서 `A`와 `B`은 balanced substring입니다. 따라서 `B` 옆에 있는 open paren의 위치는 전체 시퀀스에서 elevation이 0 미만으로 떨어지는 유일한 위치가 되며, 정확히 -1까지 떨어지게 됩니다.

`A`와 `B`이 무엇이 되어야 할지에 대한 아이디어를 얻으려면 다음 힌트를 읽어주십시오 (단서는 attention pattern plot에 있습니다!).
</details>

<details>
<summary>힌트 4</summary>

attention pattern plot을 통해, 27-31 범위의 left paren들이 38-40 위치의 token들에 기이할 정도로 강하게 attention을 기울이는 것을 볼 수 있습니다. 이는 만약 27-31 범위 내 또는 그 이후에 negative elevation이 있다면, 이 negative elevation을 감지해야 할 left bracket이 잘못 계산할 수 있음을 의미합니다. 특히, `B = ((...))` 경우, 이 left bracket이 끝부분의 right bracket들을 과하게 계산하고 `B` 시작 부분의 left bracket들에는 낮은 가중치를 두어, 실제로는 그렇지 않음에도 불구하고 시퀀스가 balanced라고 "생각"할 수 있습니다.
</details>

<details>
<summary>솔루션 (현재 알려진 최선의 adversarial example)</summary>

`A`와 `B`를 각각 길이가 $i$ 및 $38-i$인 `(((...)))` 항들의 시퀀스로 선택하십시오 (시퀀스가 negative인 단 한 지점을 제외하고 모든 곳에서 최대의 positive elevation을 갖게 하려면 `A`도 이와 같이 선택하는 것이 합리적입니다). 그런 다음, $i = 2, 4, ...\,$에 대해 최대화하십시오. 이전 힌트의 관찰 결과에 따라 당연하게도, 최선의 adversarial example들(모두 98% 이상의 balanced 확률을 가짐)은 $i=24, 26, 28, 30, 32$임을 알 수 있습니다. 이 중 가장 좋은 것은 $i=30$이며, 99.9856%의 balanced confidence를 얻습니다.

```python
def tallest_balanced_bracket(length: int) -> str:
    return "".join(["(" for _ in range(length)] + [")" for _ in range(length)])
    
example = tallest_balanced_bracket(15) + ")(" + tallest_balanced_bracket(4)
```

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/graph.png">

</details>

### 조기 닫는 괄호 처리하기

우리의 모델이 조기 닫는 괄호를 다르게 처리한다는 점을 언급했습니다. 특히 우리의 컴포넌트 중 하나는 닫는 괄호로 시작하는 모든 시퀀스를 unbalanced로 분류하는 역할을 담당합니다. 이 역할을 수행하는 컴포넌트를 찾을 수 있습니까?

<details>
<summary>힌트</summary>

이는 attention head 중 하나여야 합니다. 왜냐하면 attention head만이 시퀀스 위치 1에서 위치 0으로 정보를 이동시킬 수 있기 때문입니다 (그리고 우리가 감지하려는 실패 모드는 시퀀스의 위치 1에 닫는 괄호가 있는 경우입니다).

이전에 위치 1에서 위치 0으로 정보를 이동시키는 것으로 관찰된 attention head는 무엇이었습니까?
</details>

첫 번째 위치에 닫는 괄호가 있을 때 이 컴포넌트의 출력을 plot 할 수 있습니까? 이 컴포넌트가 이러한 동작을 담당한다는 것을 증명하고, 정확히 어떻게 발생하는지 보여줄 수 있습니까?

## 추천 캡스톤 프로젝트

### 더 많은 알고리즘 문제 시도하기

toy 모델을 해석하는 것은 TransformerLens와 기본적인 interpretability 방법론을 사용하는 자신감을 높이는 좋은 방법입니다. mechanistic interpretability의 공개 문제들 중 가장 흥미로운 카테고리는 아닐 수 있지만, 여전히 유용한 연습이 될 수 있으며, 때로는 interpretability 도구를 어떻게 사용할 수 있는지에 대한 흥미로운 새로운 통찰로 이어질 수 있습니다.

원하신다면, LeetCode에 접속하여 적절한 문제( "Easy" 섹션을 추천합니다)를 선택해 transformer를 학습시키고 그 출력을 해석해 볼 수 있습니다. 시작을 위한 몇 가지 제안은 다음과 같습니다 (일부는 LeetCode에서, 나머지는 Neel Nanda의 [open problems post](https://www.lesswrong.com/s/yivyHaCAmMJ3CqSyj/p/ejtFsvyhRkMofKAFy)에서 가져왔습니다). 제가 직접 해석해 본 것은 아니기에 추측일 뿐이지만, 대략 쉬운 순서에서 어려운 순서로 나열했습니다. 참고로, 수정을 통해 이러한 문제들을 더 쉽게 또는 더 어렵게 만들 수 있는 방법들이 있으며, 몇 가지 아이디어를 본문에 포함했습니다.

* 피보나치 스타일의 재귀 관계를 가진 시퀀스 계산 (즉, 이전 두 요소로부터 다음 요소를 예측)
* [Search Insert Position](https://leetcode.com/problems/search-insert-position/) - 타겟이 항상 리스트에 포함되는 것이 보장되는 경우 더 쉬운 버전이 됩니다 (이 경우 정렬에 대해 걱정할 필요가 없습니다). 이러한 보장이 없는 버전은 매우 다른 문제이며 훨씬 더 어려울 것입니다.
* [Is Subsequence](https://leetcode.com/problems/is-subsequence/) - 길이 1의 subsequence부터 시작하여 (이 경우 이전 문제의 쉬운 버전과 매우 유사합니다), 점차 확장해 나가야 합니다.
* [Majority Element](https://leetcode.com/problems/majority-element/) - 데이터 생성 과정을 조정하여 난이도를 변경해 볼 수 있습니다. 예를 들어, 다수 요소의 빈도에 대한 보장이 없는 시퀀스 (즉, 단순히 다른 어떤 token보다 더 많이 나타나는 token을 찾는 경우)는 훨씬 더 어려울 것입니다.
* [Number of Equivalent Domino Pairs](https://leetcode.com/problems/number-of-equivalent-domino-pairs/) - 문제를 매우 짧은 도미노 리스트로 제한하여 더 쉽게 만들 수 있습니다 (예: 도미노 2개로 시작!).
* [Longest Substring Without Repeating Characters](https://leetcode.com/problems/longest-substring-without-repeating-characters/)
* [Isomorphic Strings](https://leetcode.com/problems/isomorphic-strings/) - 첫 번째 문자열에만 중복 문자를 허용하거나, 문자열 길이 / vocabulary 크기를 제한하여 더 단순하게 만들 수 있습니다.
* [Plus One](https://leetcode.com/problems/plus-one/) - 이를 시도하기 전에 "숫자의 합" 알고리즘 문제나 이 장의 grokking 연습 문제를 살펴보는 것이 좋습니다. 이 문제를 잘 이해하면 실제로 "숫자의 합" 문제를 해석하는 단계로 나아가는 데 도움이 될 수 있습니다 (저는 이를 수행하지 않았으므로, 제가 올림 메커니즘을 깊게 파고들지 않았기 때문에 [mine](https://www.perfectlynormal.co.uk/blog-november-monthly-problem)보다 더 나은 해석을 찾아내실 가능성이 매우 높습니다).
* permutation 예측, 즉 12-token 시퀀스 `(17 3 11) (17 1 13) (11 2 4) (11 4 2)`의 마지막 3개 token을 예측하는 것입니다 (즉, 모델은 첫 번째 그룹에서 두 번째 그룹을 얻기 위해 어떤 permutation 함수가 적용되었는지 학습하고, 그 permutation을 세 번째 그룹에 적용하여 네 번째 그룹을 정확하게 예측해야 합니다). 참고로, 이 문제를 해결하려면 3개의 layer가 필요할 수 있습니다. 그 이유를 알 수 있을까요?
* [automata](https://arxiv.org/pdf/2210.10749.pdf) 작업에 대한 모델을 학습시키고 해석해 보세요. 결과가 이론과 일치합니까?
* 간단한 코드 함수의 출력 예측. 예를 들어, 다음 시퀀스에서 `1 2 4` 텍스트를 예측하는 것입니다 (이는 당연히 몇 가지 명백한 수정을 통해 더 어렵게 만들 수 있습니다. 예를 들어, 모델이 올바른 변수를 다시 attend 해야 하도록 더 많은 변수 정의를 추가하는 방식입니다):
```python
a = 1 2 3
a[2] = 4
a -> 1 2 4
```

* [this](https://jacobbrazeal.wordpress.com/2022/09/23/gpt-3-can-find-paths-up-to-7-nodes-long-in-random-graphs/)과 같은 그래프 이론 문제. 이러한 작업으로 transformer를 학습시킬 때는 입력 형식을 창의적으로 구성해야 할 수도 있습니다!

참고로, ARENA는 [monthly algorithmic problems sequence](https://arena-ch1-transformers.streamlit.app/Monthly_Algorithmic_Problems)를 운영하고 있으며, 이 시리즈의 지난 문제들을 살펴보며 아이디어를 얻을 수 있습니다. 또한 이러한 repo들을 사용하여 toy 모델에서 transformer를 구축 및 학습시키고, 특정 문제에 맞는 데이터셋을 구성하기 위한 샘플 코드를 얻을 수 있습니다.

<br>

## 추천 논문 재현

### [Causal Scrubbing](https://www.lesswrong.com/s/h95ayYYwMebGEYN5y)

Causal scrubbing은 Redwood Research에서 개발한 알고리즘으로, 계산 서브그래프가 circuit에 해당하는지 결정하기 위한 자동화된 메트릭을 생성하려고 시도합니다. 관련 읽을거리입니다:

* [Neel's dynalist notes](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=KfagbOQ29EYq3FA_OGaxZaoc) (짧은 글)
* [Causal Scrubbing: a method for rigorously testing interpretability hypotheses](https://www.lesswrong.com/posts/JvZhhzycHu2Yd57RN/causal-scrubbing-a-method-for-rigorously-testing) (알고리즘을 설명하는 전체 LessWrong 포스트)
    * Redwood의 전체 시퀀스 [here](https://www.lesswrong.com/s/h95ayYYwMebGEYN5y)도 읽어보실 수 있으며, 여기에서 그들은 이를 paren balancer에 적용하는 것을 언급합니다.
* [Practical Pitfalls of Causal Scrubbing](https://www.alignmentforum.org/posts/DFarDnQjMnjsKvW8s/practical-pitfalls-of-causal-scrubbing)

causal scrubbing 알고리즘을 작성하고, 이를 사용하여 그들의 결과를 재현할 수 있습니까? bracket classifier에 적용하기 전에 induction heads부터 시작하는 것이 좋습니다.

다음과 같은 경우에 이 재현 작업이 적합할 수 있습니다:

* 지금까지 주로 집중했던 탐색적 스타일의 작업보다 높은 수준의 엄격함을 선호하는 경우
* 이 연습 문제들을 즐겼으며, 이 bracket classifier에 의해 구현된 circuit의 종류를 잘 이해하고 있다고 느끼는 경우
* (이상적으로는) 위에서 제안한 "detecting anywhere negative failures" 보너스 연습 문제를 조사해 본 경우

### [A circuit for Python docstrings in a 4-layer attention-only transformer](https://www.lesswrong.com/posts/u6KXXmKFbXfWzoAXn/a-circuit-for-python-docstrings-in-a-4-layer-attention-only)

이 작업은 Neel Nanda의 지도 하에 SERI ML Alignment Theory Scholars Program (Winter 2022)의 일부로 수행되었습니다. IOI 논문이 어떤 의미에서 3개의 layer가 필요한 가장 단순한 종류의 circuit을 찾았던 것과 비슷하게, 이 작업은 4개의 layer가 필요한 가장 단순한 종류의 circuit을 찾고자 했습니다. 그들이 조사한 작업은 **docstring task**입니다. 다음과 같은 상황에서 파라미터를 올바른 순서로 예측할 수 있습니까? (코드는 무작위 단어를 선택하여 생성되었습니다):

```python
def port(self, load, size, files, last):
    |||oil column piece

    :param load: crime population
    :param size: unit dark
    :param
```

다음에 올 token은 ` files`이어야 하며, IOI의 경우와 마찬가지로 transformer가 이 작업을 어떻게 해결하는지 깊이 있게 분석할 수 있습니다. IOI와 달리, 우리는 코드 데이터로 학습된 4-layer transformer(GPT2-Small 아님)를 살펴보고 있으며, 이는 (circuit이 IOI보다 더 많은 수준의 composition을 가지고 있음에도 불구하고) 많은 분석을 더 깔끔하게 만들어 줍니다.

추가적인 도전을 원하신다면, 저자들의 결과를 재현하는 대신 논문 저자들이 어떤 도구를 사용했는지 보지 않고 직접 이 조사를 수행해 보십시오! 대부분은 지금까지 연습 문제에서 사용했던 도구들과 비슷할 것입니다.

다음과 같은 경우에 이 재현 작업이 적합할 수 있습니다:

* 이 연습 문제의 대부분 또는 모든 섹션을 즐겼으며, 배운 도구들을 다른 맥락에서 사용하는 연습을 하고 싶은 경우 - 구체적으로, 덜 알고리즘적이고 bracket transformer만큼 명확한 circuit을 가지고 있지 않을 수 있는 모델의 경우
* 실제 언어 모델에 조금 더 집중한 작업을 하고 싶지만, 여전히 GPT2-Small만큼 큰 모델까지는 가고 싶지 않은 경우

참고로, 이 재현 작업은 이 연습 문제들보다는 [1.3] Indirect Object Identification에 더 가깝습니다. 이 챕터를 마치기 전에 시간이 있다면, 이 연습 문제들을 먼저 시도하는 것을 권장합니다. 이 문제들이 큰 모델을 다루는 데 더 적합한 도구 세트를 갖추는 데 매우 도움이 될 것이기 때문입니다.